# BetaTour v11r2 — POP6 aday kenar mekanizma pilotu (düzeltilmiş tekrar)

Bu deney **eski v8/v9 geliştirme örneklerinin dört 10k örneğini** kullanır; bağımsız doğrulama değildir. POPMUSIC-6/LKH'nin aynı ilk denemesinden sonra BetaTour, tur sıklığı veya rastgele kontrol, aynı kaynak düğümlere ve aynı sayıda yeni yönsüz aday kenara müdahale eder. Sham aynı model çıkarımını yapar ve aramayı değiştirmez. Native POP6 ve POP7 pratik kıyaslardır. Bütün kol bütçelerine kurulum, model çıkarımı, bekleme ve LKH araması dahildir.

**v11'e göre değişiklikler:** (1) LKH her daha iyi turdan sonra aday satırlarını yeniden düzenlediği için eklenen kenarın 3. sırası artık her deneme sonunda bir *taban* olarak korunur ve kaç kez geri alındığı sayılır. (2) Aynı model girdisini alan kollar ilk saklanan model çıktısıyla karar verir; her kol yine kendi çıkarımını yapar ve süresini öder. (3) Native kolların izlemesi zaman sınırı içinde tur ayrıştırmaz. (4) Kesintide yarım kalmış tur ve olay kayıtları güvenle ele alınır. İlk v11 çalışması T4 ve 2 vCPU üzerinde koştuğu için bu tekrar en az 12 vCPU ister.

**Çalıştırma:** Colab'da **A100 (veya L4) GPU ve Yüksek RAM** seçin; defter 12'den az vCPU görürse durur. Hücreleri sırayla çalıştırın. Tamamlanan kollar `MyDrive/BetaTour/v11r2_pop6_candidate_mechanism` içinde doğrulanarak atlanır; kesilen kol yeniden çalışır. Sonuç ZIP'i sonunda tarayıcıya otomatik iner, tam denetim ZIP'i Drive'da kalır.

**Tahmini süre:** kurulum ve LKH derlemesi ≈ 3–5 dk; ısınma ve doğrulama ≈ 1 dk; dört örnek × altı kol × 120 sn = **48 dk** çözücü; kayıt ve arşiv ≈ 2–4 dk. Toplam ≈ 55–60 dk.


In [ ]:
EXPERIMENT_NAME = "v11r2_pop6_candidate_mechanism"
SEEDS = list(range(301, 307))
ARMS = ["native_pop6", "native_pop7", "sham", "random_add", "frequency_add", "beta_add"]
INSTANCE_SPECS = [
    ["uni10k", 10000, 2, "uniform", 4107, 120.0],
    ["clu10k", 10000, 2, "clustered", 4208, 120.0],
]
EXPECTED_XY_SHA256 = {
    "augment_uni10k_000": "0d5d680b465faa68d33467c4f70bb5ad1e52b25a2d444066ab922130458efcd6",
    "augment_uni10k_001": "00747175507524f481bd6e34ea55f0a5474d3dcd642a67543d871d9a64b53796",
    "augment_clu10k_000": "072fc96ec668bb9c95b3a52562f23cedb0b2e89494946d159243b638b7b3dc8e",
    "augment_clu10k_001": "965fe4315060118d313a357404f750a23ea89cc11c174cb1138a9f32b312ca22",
    "augment_uni20k_000": "d0dbad4fdd4151029d3bf59cca81c19248129c87030003d7c455e37e32b7d674",
    "augment_uni20k_001": "c35e5c135b527d91c5c9b6258320f35526c2e735a50d581c48e96150e9cb3173",
    "augment_clu20k_000": "abdd6474a4cd0e25ee394c227ef8f5c4beb5c09c8a24abddcad273d0dc39181e",
    "augment_clu20k_001": "ee3edf42dbbb94cd344d1e3bef06d5674e47330e937a8acda4fd069f4faf79ff",
}
MAX_NEW_ARMS = None  # Örn. 3: bu oturumda en fazla üç yeni kol; sonra tekrar çalıştırın.
SPARSE_FRACTION = 0.25
GRAPH_K = 16
REPO_COMMIT = "d63f7e17942a6af5cba14ad010edca1069d477b3"
CHECKPOINT_SHA256 = "2322f66c752913f6179f8c61f6b50b0f9d21426857d3ec0155e02891fb3ebd70"
LKH_TAR_SHA256 = "4c4b9e6a77e50a1eb1eb2c8cb384ed5ef64e0dc88fdbf37bd693798469e1cf33"
LKH_FINDTOUR_SHA256 = "567e72acc9b7392488dd158be4df0267560424204a0e88d2d9b62f923a7c0eee"
EXPECTED_XY_SHA256 = {name: digest for name, digest in EXPECTED_XY_SHA256.items()
                      if "10k" in name}
# Time-to-target costs: run-v10 native POPMUSIC-6 best of six (A100, 12 vCPU).
TARGET_COSTS = {"augment_clu10k_000": 33760199,
                "augment_clu10k_001": 16201186,
                "augment_uni10k_000": 72087781,
                "augment_uni10k_001": 71648570}
assert len(SEEDS) == 6 and len(EXPECTED_XY_SHA256) == 4 and len(TARGET_COSTS) == 4


In [ ]:
import base64, hashlib, importlib.util, json, math, os, platform, shutil, subprocess, sys, tarfile, time, urllib.request, uuid, zipfile
from pathlib import Path

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
for package in ("scipy", "pandas"):
    if importlib.util.find_spec(package) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)
import numpy as np
import scipy
from scipy.stats import binomtest
import pandas as pd
import torch
from IPython.display import display

assert torch.cuda.is_available(), "Colab GPU runtime seçin (A100 veya L4)."
torch.set_num_threads(1)
DEVICE = torch.device("cuda")
GPU_NAME = torch.cuda.get_device_name(0)
assert "A100" in GPU_NAME or "L4" in GPU_NAME, f"Beklenen GPU A100/L4; görünen: {GPU_NAME}"
CPU_COUNT = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else os.cpu_count()
cpu_max = Path("/sys/fs/cgroup/cpu.max")
if cpu_max.exists():
    quota, period = cpu_max.read_text().split()[:2]
    if quota != "max":
        CPU_COUNT = min(CPU_COUNT, max(1, math.floor(int(quota) / int(period))))
assert CPU_COUNT >= 12, f"En az 12 görünür vCPU gerekli; görünen: {CPU_COUNT}"
def cpu_model_name():
    for line in Path("/proc/cpuinfo").read_text().splitlines():
        if line.lower().startswith("model name"):
            return line.split(":", 1)[1].strip()
    return platform.processor() or "unknown"

WORK_ROOT = Path("/content/betatour_v11r2_candidate_work")
WORK_ROOT.mkdir(parents=True, exist_ok=True)
REPO = WORK_ROOT / "repo"
if not (REPO / ".git").exists():
    subprocess.run(["git", "clone", "--quiet", "https://github.com/kurbankotan/betatour-tsp-lkh-candidates.git", str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "fetch", "--quiet", "origin", REPO_COMMIT], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", REPO_COMMIT], check=True)
assert subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip() == REPO_COMMIT
assert hashlib.sha256((REPO / "checkpoints/betatour_v2.pt").read_bytes()).hexdigest() == CHECKPOINT_SHA256
sys.path.insert(0, str(REPO / "src"))
import tsplab as T
import tsp_model as M
MODEL, _ = M.load_model(str(REPO / "checkpoints/betatour_v2.pt"), DEVICE)
assert MODEL.cfg.get("use_ensemble") is True

from google.colab import drive
drive.mount("/content/drive")
OUTPUT_ROOT = Path("/content/drive/MyDrive/BetaTour") / EXPERIMENT_NAME
for sub in ("instances", "code", "summary", "errors", "archives"):
    (OUTPUT_ROOT / sub).mkdir(parents=True, exist_ok=True)
HARDWARE = {"gpu": GPU_NAME, "vcpus": CPU_COUNT, "cpu_model": cpu_model_name(),
            "python": platform.python_version(), "torch": torch.__version__,
            "numpy": np.__version__, "scipy": scipy.__version__,
            "pandas": pd.__version__, "cuda": torch.version.cuda}
print("Hazır:", HARDWARE, "Drive:", OUTPUT_ROOT)


Mounted at /content/drive
Hazır: {'gpu': 'NVIDIA A100-SXM4-80GB', 'vcpus': 12, 'cpu_model': 'Intel(R) Xeon(R) CPU @ 2.20GHz', 'python': '3.13.16', 'torch': '2.11.0+cu130', 'numpy': '2.1.3', 'scipy': '1.16.3', 'pandas': '2.2.3', 'cuda': '13.0'} Drive: /content/drive/MyDrive/BetaTour/v11r2_pop6_candidate_mechanism


In [ ]:
BUNDLED_FILES = {'livehook/coordinator.py': 'IiIiV2FsbC1jbG9jayBjb29yZGluYXRvciBmb3Igc2l4IHN0YXRlLXByZXNlcnZpbmcgQmV0YVRvdXIvTEtIIHByb2Nlc3Nlcy4KCk9ubHkgb25lIHByb2JsZW0gaW5zdGFuY2Ugc2hvdWxkIHVzZSBhIGdpdmVuIHdvcmsgZGlyZWN0b3J5LiBUaGUgYWN0aW9uCmNhbGxiYWNrIHJlY2VpdmVzIHRoZSBzaXggZmlyc3QgdG91cnMgYW5kIG5hdGl2ZS9saXZlIGNhbmRpZGF0ZSBzbmFwc2hvdHMuCkl0IHJldHVybnMgb25lIHBlci1ub2RlIGFjdGlvbiBsaXN0IHBlciBzZWVkOyBlYWNoIGFjdGlvbiBpcwpgYChuZWlnaGJvcl9pZF8xX2Jhc2VkX29yXzAsIHNsb3RfMV9iYXNlZF9vcl8wLCBhbHBoYSlgYC4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQganNvbgppbXBvcnQgb3MKaW1wb3J0IHN1YnByb2Nlc3MKaW1wb3J0IHRpbWUKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHR5cGluZyBpbXBvcnQgQ2FsbGFibGUsIFNlcXVlbmNlCgoKQGRhdGFjbGFzcwpjbGFzcyBTbmFwc2hvdDoKICAgIHNlZWQ6IGludAogICAgZmlyc3RfdG91cjogbGlzdFtpbnRdCiAgICBuYXRpdmVfY2FuZGlkYXRlczogbGlzdFtsaXN0W3R1cGxlW2ludCwgaW50XV1dCiAgICBsaXZlX2NhbmRpZGF0ZXM6IGxpc3RbbGlzdFt0dXBsZVtpbnQsIGludF1dXQoKCmRlZiByZWFkX3RvdXIocGF0aDogUGF0aCkgLT4gbGlzdFtpbnRdOgogICAgd29yZHMgPSBbaW50KHgpIGZvciB4IGluIHBhdGgucmVhZF90ZXh0KCkuc3BsaXQoKV0KICAgIGlmIG5vdCB3b3JkcyBvciBsZW4od29yZHMpICE9IHdvcmRzWzBdICsgMToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiTWFsZm9ybWVkIHRvdXI6IHtwYXRofSIpCiAgICBuID0gd29yZHNbMF0KICAgIHRvdXIgPSB3b3Jkc1sxOl0KICAgIGlmIHNvcnRlZCh0b3VyKSAhPSBsaXN0KHJhbmdlKDEsIG4gKyAxKSk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIlRvdXIgaXMgbm90IGEgcGVybXV0YXRpb246IHtwYXRofSIpCiAgICByZXR1cm4gdG91cgoKCmRlZiByZWFkX2NhbmRpZGF0ZXMocGF0aDogUGF0aCkgLT4gbGlzdFtsaXN0W3R1cGxlW2ludCwgaW50XV1dOgogICAgbGluZXMgPSBwYXRoLnJlYWRfdGV4dCgpLnNwbGl0bGluZXMoKQogICAgaWYgbm90IGxpbmVzOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJFbXB0eSBjYW5kaWRhdGUgc25hcHNob3Q6IHtwYXRofSIpCiAgICBuID0gaW50KGxpbmVzWzBdKQogICAgaWYgbGVuKGxpbmVzKSAhPSBuICsgMToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiQ2FuZGlkYXRlIHNuYXBzaG90IGhhcyB7bGVuKGxpbmVzKS0xfSByb3dzLCBuZWVkIHtufSIpCiAgICByb3dzOiBsaXN0W2xpc3RbdHVwbGVbaW50LCBpbnRdXV0gPSBbXQogICAgZm9yIGV4cGVjdGVkX2lkLCBsaW5lIGluIGVudW1lcmF0ZShsaW5lc1sxOl0sIDEpOgogICAgICAgIG51bXMgPSBbaW50KHgpIGZvciB4IGluIGxpbmUuc3BsaXQoKV0KICAgICAgICBpZiBsZW4obnVtcykgPCAyIG9yIG51bXNbMF0gIT0gZXhwZWN0ZWRfaWQgb3IgbGVuKG51bXMpICE9IDIgKyAyICogbnVtc1sxXToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIk1hbGZvcm1lZCBjYW5kaWRhdGUgcm93IHtleHBlY3RlZF9pZH06IHtwYXRofSIpCiAgICAgICAgcm93ID0gWyhudW1zW2ldLCBudW1zW2kgKyAxXSkgZm9yIGkgaW4gcmFuZ2UoMiwgbGVuKG51bXMpLCAyKV0KICAgICAgICBpZiBhbnkodiA8IDEgb3IgdiA+IG4gb3IgdiA9PSBleHBlY3RlZF9pZCBmb3IgdiwgXyBpbiByb3cpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiSW52YWxpZCBuZWlnaGJvciBpbiByb3cge2V4cGVjdGVkX2lkfToge3BhdGh9IikKICAgICAgICByb3dzLmFwcGVuZChyb3cpCiAgICByZXR1cm4gcm93cwoKCmRlZiBub29wX2FjdGlvbnMobjogaW50KSAtPiBsaXN0W3R1cGxlW2ludCwgaW50LCBpbnRdXToKICAgIHJldHVybiBbKDAsIDAsIDUwMCldICogbgoKCmRlZiB3cml0ZV9hY3Rpb25zKHBhdGg6IFBhdGgsIGFjdGlvbnM6IFNlcXVlbmNlW3R1cGxlW2ludCwgaW50LCBpbnRdXSkgLT4gTm9uZToKICAgIG4gPSBsZW4oYWN0aW9ucykKICAgIGxpbmVzID0gW3N0cihuKV0KICAgIGZvciBzb3VyY2VfaWQsIChuZWlnaGJvciwgc2xvdCwgYWxwaGEpIGluIGVudW1lcmF0ZShhY3Rpb25zLCAxKToKICAgICAgICBpZiBub3QgKDAgPD0gbmVpZ2hib3IgPD0gbiBhbmQgMCA8PSBzbG90IDw9IG4gKyAyIGFuZCAwIDw9IGFscGhhIDwgMioqMzEgLSAxKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIkludmFsaWQgYWN0aW9uIGF0IHNvdXJjZSB7c291cmNlX2lkfSIpCiAgICAgICAgaWYgbmVpZ2hib3IgPT0gc291cmNlX2lkOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiU2VsZi1lZGdlIGF0IHNvdXJjZSB7c291cmNlX2lkfSIpCiAgICAgICAgbGluZXMuYXBwZW5kKGYie3NvdXJjZV9pZH0ge25laWdoYm9yfSB7c2xvdH0ge2FscGhhfSIpCiAgICB0ZW1wb3JhcnkgPSBwYXRoLndpdGhfc3VmZml4KHBhdGguc3VmZml4ICsgIi50bXAiKQogICAgdGVtcG9yYXJ5LndyaXRlX3RleHQoIlxuIi5qb2luKGxpbmVzKSArICJcbiIpCiAgICBvcy5yZXBsYWNlKHRlbXBvcmFyeSwgcGF0aCkKCgpkZWYgX3BhcmFtZXRlcl9maWxlKHBhdGg6IFBhdGgsIHByb2JsZW06IFBhdGgsIHNlZWQ6IGludCwgd2FsbF9idWRnZXRfczogZmxvYXQsCiAgICAgICAgICAgICAgICAgICAgbWF4X3RyaWFsczogaW50LCBleHRyYTogZGljdFtzdHIsIHN0ciB8IGludCB8IGZsb2F0XSkgLT4gTm9uZToKICAgIHBhcmFtczogZGljdFtzdHIsIHN0ciB8IGludCB8IGZsb2F0XSA9IHsKICAgICAgICAiUFJPQkxFTV9GSUxFIjogc3RyKHByb2JsZW0ucmVzb2x2ZSgpKSwKICAgICAgICAiQ0FORElEQVRFX1NFVF9UWVBFIjogIlBPUE1VU0lDIiwKICAgICAgICAiTUFYX0NBTkRJREFURVMiOiA1LAogICAgICAgICJTVUJHUkFESUVOVCI6ICJOTyIsCiAgICAgICAgIlNFRUQiOiBzZWVkLAogICAgICAgICJSVU5TIjogMSwKICAgICAgICAiTUFYX1RSSUFMUyI6IG1heF90cmlhbHMsCiAgICAgICAgIyBMS0gncyBUSU1FX0xJTUlUIG1lYXN1cmVzIHByb2Nlc3MgQ1BVIHRpbWUuIFRoZSBjb29yZGluYXRvciBiZWxvdwogICAgICAgICMgZW5mb3JjZXMgdGhlIGFjdHVhbCBlbmQtdG8tZW5kIHdhbGwgYnVkZ2V0LgogICAgICAgICJUSU1FX0xJTUlUIjogbWF4KDEwLjAsIDEwLjAgKiB3YWxsX2J1ZGdldF9zKSwKICAgICAgICAiVFJBQ0VfTEVWRUwiOiAwLAogICAgICAgICJTVE9QX0FUX09QVElNVU0iOiAiTk8iLAogICAgfQogICAgcGFyYW1zLnVwZGF0ZShleHRyYSkKICAgIHBhdGgud3JpdGVfdGV4dCgiXG4iLmpvaW4oZiJ7a2V5fSA9IHt2YWx1ZX0iIGZvciBrZXksIHZhbHVlIGluIHBhcmFtcy5pdGVtcygpKSArICJcbiIpCgoKZGVmIHJ1bl9zaXgoCiAgICBsa2hfYmluYXJ5OiBQYXRoLAogICAgcHJvYmxlbTogUGF0aCwKICAgIHdvcmtfZGlyOiBQYXRoLAogICAgc2VlZHM6IFNlcXVlbmNlW2ludF0sCiAgICB3YWxsX2J1ZGdldF9zOiBmbG9hdCwKICAgIGFjdGlvbl9mbjogQ2FsbGFibGVbW2xpc3RbU25hcHNob3RdXSwgU2VxdWVuY2VbU2VxdWVuY2VbdHVwbGVbaW50LCBpbnQsIGludF1dXV0sCiAgICAqLAogICAgbWF4X3RyaWFsczogaW50ID0gMTAwXzAwMF8wMDAsCiAgICBleHRyYV9wYXJhbXM6IGRpY3Rbc3RyLCBzdHIgfCBpbnQgfCBmbG9hdF0gfCBOb25lID0gTm9uZSwKICAgIGJhcnJpZXJfbmFtZTogc3RyID0gImZpcnN0LnRvdXIiLAogICAgaG9va19hZnRlcl9zOiBmbG9hdCA9IDAuMCwKKSAtPiBkaWN0OgogICAgIiIiUnVuIG9uZSBpbnN0YW5jZSB3aXRoIHNpeCBzZWVkczsgY2hhcmdlIHN0YXJ0dXAsIGJhcnJpZXIgYW5kIHBvbGljeSB0aW1lLgoKICAgIGB3b3JrX2RpcmAgbXVzdCBiZSBuZXcgdG8gYXZvaWQgc3RhbGUgYWN0aW9uIGZpbGVzLiBSdW5zIG1heSBlbmQgYmVmb3JlIHRoZQogICAgZGVhZGxpbmUgd2hlbiBgbWF4X3RyaWFsc2AgaXMgc21hbGwuIFJlc3VsdHMgaW5jbHVkZSBhbGwgaG9vayBmaWxlcyBzbyB0aGUKICAgIGNhbGxlciBjYW4gYXVkaXQgdGhlIGFjdHVhbCBjYW5kaWRhdGUgYXJyYXlzIGFuZCBmaXJzdCB0b3Vycy4KICAgICIiIgogICAgaWYgbGVuKHNlZWRzKSAhPSA2IG9yIGxlbihzZXQoc2VlZHMpKSAhPSA2OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkV4YWN0bHkgc2l4IGRpc3RpbmN0IHNlZWRzIGFyZSByZXF1aXJlZCIpCiAgICBpZiB3YWxsX2J1ZGdldF9zIDw9IDAgb3IgbWF4X3RyaWFscyA8IDE6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiUG9zaXRpdmUgYnVkZ2V0IGFuZCB0cmlhbHMgYXJlIHJlcXVpcmVkIikKICAgIGlmIGJhcnJpZXJfbmFtZSBub3QgaW4gKCJmaXJzdC50b3VyIiwgImJhcnJpZXIudG91ciIpIG9yIGhvb2tfYWZ0ZXJfcyA8IDA6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiSW52YWxpZCBiYXJyaWVyIG1hcmtlciBvciBkZWxheSIpCiAgICBpZiB3b3JrX2Rpci5leGlzdHMoKToKICAgICAgICByYWlzZSBGaWxlRXhpc3RzRXJyb3IoZiJVc2UgYSBmcmVzaCBydW4gZGlyZWN0b3J5OiB7d29ya19kaXJ9IikKICAgIHdvcmtfZGlyLm1rZGlyKHBhcmVudHM9VHJ1ZSkKICAgIHByb2Nlc3NlczogbGlzdFtzdWJwcm9jZXNzLlBvcGVuXSA9IFtdCiAgICBvcGVuX2xvZ3M6IGxpc3QgPSBbXQogICAgc3RhcnRfbW9ubyA9IHRpbWUubW9ub3RvbmljKCkKICAgIHN0YXJ0X2Vwb2NoID0gdGltZS50aW1lKCkKICAgIGRlYWRsaW5lID0gc3RhcnRfbW9ubyArIHdhbGxfYnVkZ2V0X3MKICAgIHRyeToKICAgICAgICBmb3Igc2VlZCBpbiBzZWVkczoKICAgICAgICAgICAgc2VlZF9kaXIgPSB3b3JrX2RpciAvIGYic2VlZF97c2VlZH0iCiAgICAgICAgICAgIHNlZWRfZGlyLm1rZGlyKCkKICAgICAgICAgICAgcGFyYW1zID0gc2VlZF9kaXIgLyAicnVuLnBhciIKICAgICAgICAgICAgX3BhcmFtZXRlcl9maWxlKHBhcmFtcywgcHJvYmxlbSwgc2VlZCwgd2FsbF9idWRnZXRfcywgbWF4X3RyaWFscywKICAgICAgICAgICAgICAgICAgICAgICAgICAgIGV4dHJhX3BhcmFtcyBvciB7fSkKICAgICAgICAgICAgc3Rkb3V0ID0gKHNlZWRfZGlyIC8gInN0ZG91dC50eHQiKS5vcGVuKCJ3IikKICAgICAgICAgICAgc3RkZXJyID0gKHNlZWRfZGlyIC8gInN0ZGVyci50eHQiKS5vcGVuKCJ3IikKICAgICAgICAgICAgb3Blbl9sb2dzLmV4dGVuZCgoc3Rkb3V0LCBzdGRlcnIpKQogICAgICAgICAgICBlbnYgPSBkaWN0KG9zLmVudmlyb24pCiAgICAgICAgICAgIGVudlsiQkVUQVRPVVJfSE9PS19ESVIiXSA9IHN0cihzZWVkX2Rpci5yZXNvbHZlKCkpCiAgICAgICAgICAgIGVudlsiQkVUQVRPVVJfSE9PS19USU1FT1VUX1MiXSA9IHN0cihtYXgoNS4wLCB3YWxsX2J1ZGdldF9zICsgNS4wKSkKICAgICAgICAgICAgZW52WyJCRVRBVE9VUl9IT09LX0FGVEVSX1MiXSA9IHN0cihob29rX2FmdGVyX3MpCiAgICAgICAgICAgIHByb2Nlc3Nlcy5hcHBlbmQoc3VicHJvY2Vzcy5Qb3BlbigKICAgICAgICAgICAgICAgIFtzdHIobGtoX2JpbmFyeS5yZXNvbHZlKCkpLCBzdHIocGFyYW1zLnJlc29sdmUoKSldLAogICAgICAgICAgICAgICAgY3dkPXNlZWRfZGlyLCBzdGRvdXQ9c3Rkb3V0LCBzdGRlcnI9c3RkZXJyLCBlbnY9ZW52LAogICAgICAgICAgICApKQogICAgICAgIGZpcnN0X3BhdGhzID0gW3dvcmtfZGlyIC8gZiJzZWVkX3tzZWVkfSIgLyBiYXJyaWVyX25hbWUgZm9yIHNlZWQgaW4gc2VlZHNdCiAgICAgICAgd2hpbGUgbm90IGFsbChwYXRoLmV4aXN0cygpIGZvciBwYXRoIGluIGZpcnN0X3BhdGhzKToKICAgICAgICAgICAgZmFpbHVyZXMgPSBbcC5yZXR1cm5jb2RlIGZvciBwIGluIHByb2Nlc3NlcyBpZiBwLnBvbGwoKSBub3QgaW4gKE5vbmUsIDApXQogICAgICAgICAgICBpZiBmYWlsdXJlczoKICAgICAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIkxLSCBleGl0ZWQgYmVmb3JlIGZpcnN0LXRvdXIgYmFycmllcjoge2ZhaWx1cmVzfSIpCiAgICAgICAgICAgIGlmIHRpbWUubW9ub3RvbmljKCkgPj0gZGVhZGxpbmU6CiAgICAgICAgICAgICAgICBicmVhawogICAgICAgICAgICB0aW1lLnNsZWVwKDAuMDIpCiAgICAgICAgc25hcHNob3RzOiBsaXN0W1NuYXBzaG90XSA9IFtdCiAgICAgICAgaWYgYWxsKHBhdGguZXhpc3RzKCkgZm9yIHBhdGggaW4gZmlyc3RfcGF0aHMpIGFuZCB0aW1lLm1vbm90b25pYygpIDwgZGVhZGxpbmU6CiAgICAgICAgICAgIGZvciBzZWVkIGluIHNlZWRzOgogICAgICAgICAgICAgICAgYmFzZSA9IHdvcmtfZGlyIC8gZiJzZWVkX3tzZWVkfSIKICAgICAgICAgICAgICAgIHNuYXBzaG90cy5hcHBlbmQoU25hcHNob3QoCiAgICAgICAgICAgICAgICAgICAgc2VlZD1zZWVkLAogICAgICAgICAgICAgICAgICAgIGZpcnN0X3RvdXI9cmVhZF90b3VyKGJhc2UgLyBiYXJyaWVyX25hbWUpLAogICAgICAgICAgICAgICAgICAgIG5hdGl2ZV9jYW5kaWRhdGVzPXJlYWRfY2FuZGlkYXRlcyhiYXNlIC8gIm5hdGl2ZS5jYW5kaWRhdGVzIiksCiAgICAgICAgICAgICAgICAgICAgbGl2ZV9jYW5kaWRhdGVzPXJlYWRfY2FuZGlkYXRlcyhiYXNlIC8gImxpdmUuY2FuZGlkYXRlcyIpLAogICAgICAgICAgICAgICAgKSkKICAgICAgICAgICAgYWN0aW9ucyA9IGFjdGlvbl9mbihzbmFwc2hvdHMpCiAgICAgICAgICAgIGlmIGxlbihhY3Rpb25zKSAhPSBsZW4oc2VlZHMpOgogICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiUG9saWN5IG11c3QgcmV0dXJuIG9uZSBhY3Rpb24gbWF0cml4IHBlciBzZWVkIikKICAgICAgICAgICAgaWYgdGltZS5tb25vdG9uaWMoKSA8IGRlYWRsaW5lOgogICAgICAgICAgICAgICAgZm9yIHNlZWQsIHNuYXBzaG90LCBhY3Rpb24gaW4gemlwKHNlZWRzLCBzbmFwc2hvdHMsIGFjdGlvbnMpOgogICAgICAgICAgICAgICAgICAgIGlmIGxlbihhY3Rpb24pICE9IGxlbihzbmFwc2hvdC5maXJzdF90b3VyKToKICAgICAgICAgICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIlBvbGljeSBkaW1lbnNpb24gbWlzbWF0Y2ggZm9yIHNlZWQge3NlZWR9IikKICAgICAgICAgICAgICAgICAgICB3cml0ZV9hY3Rpb25zKHdvcmtfZGlyIC8gZiJzZWVkX3tzZWVkfSIgLyAiYWN0aW9uLnR4dCIsIGFjdGlvbikKICAgICAgICB3aGlsZSBhbnkocC5wb2xsKCkgaXMgTm9uZSBmb3IgcCBpbiBwcm9jZXNzZXMpIGFuZCB0aW1lLm1vbm90b25pYygpIDwgZGVhZGxpbmU6CiAgICAgICAgICAgIHRpbWUuc2xlZXAoMC4wMikKICAgIGZpbmFsbHk6CiAgICAgICAgZm9yIHByb2Nlc3MgaW4gcHJvY2Vzc2VzOgogICAgICAgICAgICBpZiBwcm9jZXNzLnBvbGwoKSBpcyBOb25lOgogICAgICAgICAgICAgICAgcHJvY2Vzcy50ZXJtaW5hdGUoKQogICAgICAgIGZvciBwcm9jZXNzIGluIHByb2Nlc3NlczoKICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgcHJvY2Vzcy53YWl0KHRpbWVvdXQ9MikKICAgICAgICAgICAgZXhjZXB0IHN1YnByb2Nlc3MuVGltZW91dEV4cGlyZWQ6CiAgICAgICAgICAgICAgICBwcm9jZXNzLmtpbGwoKQogICAgICAgICAgICAgICAgcHJvY2Vzcy53YWl0KCkKICAgICAgICBmb3Igc3RyZWFtIGluIG9wZW5fbG9nczoKICAgICAgICAgICAgc3RyZWFtLmNsb3NlKCkKICAgIHJlc3VsdCA9IHsKICAgICAgICAic2VlZHMiOiBsaXN0KHNlZWRzKSwKICAgICAgICAid2FsbF9idWRnZXRfcyI6IHdhbGxfYnVkZ2V0X3MsCiAgICAgICAgIndhbGxfZWxhcHNlZF9zIjogdGltZS5tb25vdG9uaWMoKSAtIHN0YXJ0X21vbm8sCiAgICAgICAgInN0YXJ0X2Vwb2NoX3MiOiBzdGFydF9lcG9jaCwKICAgICAgICAiZGVhZGxpbmVfZXBvY2hfcyI6IHN0YXJ0X2Vwb2NoICsgd2FsbF9idWRnZXRfcywKICAgICAgICAiYmFycmllcl9jb21wbGV0ZWQiOiBsZW4oc25hcHNob3RzKSA9PSA2LAogICAgICAgICJydW5zIjogW10sCiAgICB9CiAgICBmb3Igc2VlZCwgcHJvY2VzcyBpbiB6aXAoc2VlZHMsIHByb2Nlc3Nlcyk6CiAgICAgICAgYmFzZSA9IHdvcmtfZGlyIC8gZiJzZWVkX3tzZWVkfSIKICAgICAgICBiZXN0ID0gYmFzZSAvICJiZXN0LnRvdXIiCiAgICAgICAgcmVzdWx0WyJydW5zIl0uYXBwZW5kKHsKICAgICAgICAgICAgInNlZWQiOiBzZWVkLAogICAgICAgICAgICAicmV0dXJuY29kZSI6IHByb2Nlc3MucmV0dXJuY29kZSwKICAgICAgICAgICAgImZpcnN0X3RvdXIiOiBzdHIoYmFzZSAvIGJhcnJpZXJfbmFtZSkgaWYgKGJhc2UgLyBiYXJyaWVyX25hbWUpLmV4aXN0cygpIGVsc2UgTm9uZSwKICAgICAgICAgICAgImJlc3RfdG91ciI6IHN0cihiZXN0KSBpZiBiZXN0LmV4aXN0cygpIGVsc2UgTm9uZSwKICAgICAgICAgICAgImJlc3RfZXZlbnRzIjogc3RyKGJhc2UgLyAiYmVzdC5ldmVudHMiKSBpZiAoYmFzZSAvICJiZXN0LmV2ZW50cyIpLmV4aXN0cygpIGVsc2UgTm9uZSwKICAgICAgICAgICAgIm5hdGl2ZV9jYW5kaWRhdGVzIjogc3RyKGJhc2UgLyAibmF0aXZlLmNhbmRpZGF0ZXMiKSBpZiAoYmFzZSAvICJuYXRpdmUuY2FuZGlkYXRlcyIpLmV4aXN0cygpIGVsc2UgTm9uZSwKICAgICAgICAgICAgImxpdmVfY2FuZGlkYXRlcyI6IHN0cihiYXNlIC8gImxpdmUuY2FuZGlkYXRlcyIpIGlmIChiYXNlIC8gImxpdmUuY2FuZGlkYXRlcyIpLmV4aXN0cygpIGVsc2UgTm9uZSwKICAgICAgICAgICAgInBvc3RfY2FuZGlkYXRlcyI6IHN0cihiYXNlIC8gInBvc3QuY2FuZGlkYXRlcyIpIGlmIChiYXNlIC8gInBvc3QuY2FuZGlkYXRlcyIpLmV4aXN0cygpIGVsc2UgTm9uZSwKICAgICAgICAgICAgImFwcGxpZWRfc3RhdHMiOiBzdHIoYmFzZSAvICJhcHBsaWVkLnN0YXRzIikgaWYgKGJhc2UgLyAiYXBwbGllZC5zdGF0cyIpLmV4aXN0cygpIGVsc2UgTm9uZSwKICAgICAgICB9KQogICAgKHdvcmtfZGlyIC8gInJ1bl9zdW1tYXJ5Lmpzb24iKS53cml0ZV90ZXh0KGpzb24uZHVtcHMocmVzdWx0LCBpbmRlbnQ9MikgKyAiXG4iKQogICAgcmV0dXJuIHJlc3VsdAo=', 'v11_candidate/BetaTourSearchHook.h': '...', 'v11_candidate/BetaTourVisit.h': 'LyogU2hhcmVkLCBsb3ctb3ZlcmhlYWQgY291bnRlcnMgZm9yIGFjdGlvbi1lZGdlIGVudW1lcmF0aW9uIGluIExLIGxvb3BzLiAqLwojaWZuZGVmIEJFVEFUT1VSX1ZJU0lUX0gKI2RlZmluZSBCRVRBVE9VUl9WSVNJVF9ICgpleHRlcm4gaW50ICpCVF9BY3Rpb25UbzsKZXh0ZXJuIHVuc2lnbmVkIGxvbmcgbG9uZyAqQlRfQWN0aW9uVmlzaXRzOwpleHRlcm4gdW5zaWduZWQgbG9uZyBsb25nICpCVF9BY3Rpb25GaXJzdEdhdGVQYXNzZXM7CmV4dGVybiB1bnNpZ25lZCBsb25nIGxvbmcgKkJUX0FjdGlvblBvc2l0aXZlRzE7CmV4dGVybiB1bnNpZ25lZCBsb25nIGxvbmcgKkJUX0FjdGlvbkFjY2VwdGVkRW50cmllczsKZXh0ZXJuIHVuc2lnbmVkIGxvbmcgbG9uZyBCVF9Ub3RhbENhbmRpZGF0ZUVudW1lcmF0aW9uczsKZXh0ZXJuIHVuc2lnbmVkIGxvbmcgbG9uZyBCVF9Ub3RhbEFjdGlvbkVudW1lcmF0aW9uczsKZXh0ZXJuIHVuc2lnbmVkIGxvbmcgbG9uZyBCVF9Ub3RhbFRvcExldmVsRW51bWVyYXRpb25zOwpleHRlcm4gdW5zaWduZWQgbG9uZyBsb25nIEJUX1RvdGFsVG9wTGV2ZWxBY3Rpb25FbnVtZXJhdGlvbnM7CmV4dGVybiB1bnNpZ25lZCBsb25nIGxvbmcgQlRfVG90YWxBY3Rpb25GaXJzdEdhdGVQYXNzZXM7CmV4dGVybiB1bnNpZ25lZCBsb25nIGxvbmcgQlRfVG90YWxBY3Rpb25Qb3NpdGl2ZUcxOwpleHRlcm4gdW5zaWduZWQgbG9uZyBsb25nIEJUX1RvdGFsQWN0aW9uQWNjZXB0ZWRFbnRyaWVzOwp2b2lkIEJUX0ZsdXNoVGVsZW1ldHJ5UGVyaW9kaWModm9pZCk7CgovKiBDb3VudHMgYSBkaXJlY3RlZCBjYW5kaWRhdGUgd2hlbiBhIHBhdGNoZWQgc2VhcmNoIGxvb3AgZW51bWVyYXRlcyBpdC4KICogRWxpZ2liaWxpdHkgYW5kIGdhaW4gZmlsdGVycyBtYXkgc3RpbGwgcmVqZWN0IGl0IGxhdGVyLiAqLwpzdGF0aWMgaW5saW5lIHZvaWQgQlRfTm90ZUNhbmRpZGF0ZShOb2RlICpGcm9tLCBOb2RlICpUbykKewogICAgaWYgKCFCVF9BY3Rpb25UbyB8fCAhVG8pCiAgICAgICAgcmV0dXJuOwogICAgQlRfVG90YWxDYW5kaWRhdGVFbnVtZXJhdGlvbnMrKzsKICAgIGlmIChCVF9BY3Rpb25Ub1tGcm9tLT5JZF0gPT0gVG8tPklkKSB7CiAgICAgICAgQlRfVG90YWxBY3Rpb25FbnVtZXJhdGlvbnMrKzsKICAgICAgICBCVF9BY3Rpb25WaXNpdHNbRnJvbS0+SWRdKys7CiAgICB9CiAgICBpZiAoKEJUX1RvdGFsQ2FuZGlkYXRlRW51bWVyYXRpb25zICYgNjU1MzVVTEwpID09IDApCiAgICAgICAgQlRfRmx1c2hUZWxlbWV0cnlQZXJpb2RpYygpOwp9CgovKiBUaGlzIGlzIHRoZSBmaXJzdCB0Mi0+dDMgY2hvaWNlIGluIEJlc3QyLzMvNC81L0svU3BlY2lhbE9wdE1vdmUgb25seS4gKi8Kc3RhdGljIGlubGluZSB2b2lkIEJUX05vdGVUb3BMZXZlbEVudW1lcmF0ZWQoTm9kZSAqRnJvbSwgTm9kZSAqVG8pCnsKICAgIGlmICghQlRfQWN0aW9uVG8pCiAgICAgICAgcmV0dXJuOwogICAgQlRfVG90YWxUb3BMZXZlbEVudW1lcmF0aW9ucysrOwogICAgaWYgKEJUX0FjdGlvblRvW0Zyb20tPklkXSA9PSBUby0+SWQpCiAgICAgICAgQlRfVG90YWxUb3BMZXZlbEFjdGlvbkVudW1lcmF0aW9ucysrOwp9CgovKiBDYWxsZWQgYWZ0ZXIgdGhhdCBsb29wJ3MgZmlyc3QgYWRqYWNlbmN5L0cxIGdhdGUuIFRoaXMgaXMgbm90IHRoZSBmdWxsCiAqIG11bHRpLXN0ZXAgZ2FpbiBmaWx0ZXIsIGFuZCBHMSA+IDAgaXMgdHJhY2tlZCBzZXBhcmF0ZWx5LiAqLwpzdGF0aWMgaW5saW5lIHZvaWQgQlRfTm90ZVRvcExldmVsUGFzcyhOb2RlICpGcm9tLCBOb2RlICpUbywgR2FpblR5cGUgRzEpCnsKICAgIGlmICghQlRfQWN0aW9uVG8gfHwgQlRfQWN0aW9uVG9bRnJvbS0+SWRdICE9IFRvLT5JZCkKICAgICAgICByZXR1cm47CiAgICBCVF9BY3Rpb25GaXJzdEdhdGVQYXNzZXNbRnJvbS0+SWRdKys7CiAgICBCVF9Ub3RhbEFjdGlvbkZpcnN0R2F0ZVBhc3NlcysrOwogICAgaWYgKEcxID4gMCkgewogICAgICAgIEJUX0FjdGlvblBvc2l0aXZlRzFbRnJvbS0+SWRdKys7CiAgICAgICAgQlRfVG90YWxBY3Rpb25Qb3NpdGl2ZUcxKys7CiAgICB9Cn0KCi8qIFN0b3JlVG91ciBjYWxscyB0aGlzIGJlZm9yZSByZXBsYWNpbmcgT2xkUHJlZC9PbGRTdWMgb24gYWZmZWN0ZWQgbm9kZXMuCiAqIEl0IGNvdW50cyBhbiBhY3Rpb24gZWRnZSBuZXdseSBwcmVzZW50IGluIGEgbG9jYWxseSBhY2NlcHRlZCBMSyB0b3VyLiAqLwpzdGF0aWMgaW5saW5lIHZvaWQgQlRfTm90ZUFjY2VwdGVkRWRnZShOb2RlICpGcm9tKQp7CiAgICBpbnQgVG87CiAgICBpZiAoIUJUX0FjdGlvblRvIHx8ICEoVG8gPSBCVF9BY3Rpb25Ub1tGcm9tLT5JZF0pKQogICAgICAgIHJldHVybjsKICAgIGlmICgoRnJvbS0+UHJlZC0+SWQgPT0gVG8gfHwgRnJvbS0+U3VjLT5JZCA9PSBUbykgJiYKICAgICAgICBGcm9tLT5PbGRQcmVkLT5JZCAhPSBUbyAmJiBGcm9tLT5PbGRTdWMtPklkICE9IFRvKSB7CiAgICAgICAgQlRfQWN0aW9uQWNjZXB0ZWRFbnRyaWVzW0Zyb20tPklkXSsrOwogICAgICAgIEJUX1RvdGFsQWN0aW9uQWNjZXB0ZWRFbnRyaWVzKys7CiAgICB9Cn0KCiNlbmRpZgo=', 'v11_candidate/build_search_hook.py': 'IiIiQnVpbGQgdGhlIHYxMSBpbnN0cnVtZW50ZWQgZmlyc3QtdHJpYWwgaG9vayBmcm9tIHByaXN0aW5lIExLSC0zLjAuMTQuCgpPbmx5IG91ciB0d28gaGVhZGVycyBhbmQgZXhhY3Qtc291cmNlIHBhdGNoZXMgYXJlIHNoaXBwZWQuIFVwc3RyZWFtIExLSCBpcwpkb3dubG9hZGVkIHNlcGFyYXRlbHkgYnkgdGhlIG5vdGVib29rIGFuZCB2ZXJpZmllZCBiZWZvcmUgbW9kaWZpY2F0aW9uLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBhcmdwYXJzZQppbXBvcnQgaGFzaGxpYgppbXBvcnQganNvbgppbXBvcnQgcmUKaW1wb3J0IHNodXRpbAppbXBvcnQgc3VicHJvY2Vzcwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCgpVUFNUUkVBTV9TSEEyNTYgPSB7CiAgICAiRmluZFRvdXIuYyI6ICI1NjdlNzJhY2M5YjczOTI0ODhkZDE1OGJlNGRmMDI2NzU2MDQyNDIwNGEwZTg4ZDJkOWI2MmY5MjNhN2MwZWVlIiwKICAgICJCZXN0Mk9wdE1vdmUuYyI6ICIxNzVhZGUwMDhhYTgyNWRiODk2YTNmZDEyMWUxNDY5MWJlYzk4M2E0YjUzOWNiNjlhOWQ2MDgyZmZhMTE3ZWFkIiwKICAgICJCZXN0M09wdE1vdmUuYyI6ICIzNWFiZmU5YzJlNjZmYjA4ZTNjMTUyOTg1NTZjZjUzODAxMDU4YzUzZTY1NTFiMDVjMDViNDg0ODFmYzAzMzVmIiwKICAgICJCZXN0NE9wdE1vdmUuYyI6ICI3MDhhYzU3NTY2ZmJiZDIzNThlMjMwMzZiOTg3MGU1Yzc3ZjNlZmQ2MDBiMmYyMWI4MGVlZDNjYjk4MTExYmY5IiwKICAgICJCZXN0NU9wdE1vdmUuYyI6ICIwOWE3Yjg1ZDIwMGEzYTYxZjExMDZiNDM1MzFlYTY2MzZjNWFhZDM5N2JmOWVlODY0YmIzNjkyZmNlNjcwYmNiIiwKICAgICJCZXN0S09wdE1vdmUuYyI6ICI2MDg4ZGJkM2U3YTM2ZjQyYTI0ZGYxN2VkZDZhZDJlZTc3YjRiMjIwN2JlZjU0NTViOWI4ZmNkYzM4YjMyYzc2IiwKICAgICJCZXN0U3BlY2lhbE9wdE1vdmUuYyI6ICJlZmJkZmI1YWU5YTEyYjgxNmUzNzM0OTAxOTY5ZTU5ZjMxYmYwMTlkMDcxM2IyM2M4YmQxOWU0YWFmMDZkMzAxIiwKICAgICJHYWluMjMuYyI6ICI4NTNkODgxYjk5YzYwMzVlMGMwOGVkYjU4ZmQ1ZDRlNDQzN2Y4OGMxNTQxNWQ4ZDE4MmRiMTk0MjdlZDlkM2FiIiwKICAgICJCcmlkZ2VHYWluLmMiOiAiZDYyNWU0ZDYzYzE5MjAzN2ZjNTQwZjcyNDlkM2VkNjJhNWY4ZjdlMmU0NGYxMDhkNWFiNzBjOWIzMTRiYjk2MyIsCiAgICAiUGF0Y2hDeWNsZXMuYyI6ICIyNTNlNDdiMWJiNjRjMTFkNzZlMzcyNDEyYjgzMzk0MTFlNGQwNzkyNDExYmU5MzdlNDY2Njc4OTViZjYwYWJjIiwKICAgICJTdG9yZVRvdXIuYyI6ICI2ODMyNTUzNjBiOGQ5ZDcxZjM5MDQ4OTgwNTJjYTQ3YWUxYmVmNWZlYTc3YzIzMTFhMjBkMTY5NjcxMTY2Zjc0IiwKfQoKTU9WRV9MT09QX0NPVU5UUyA9IHsKICAgICJCZXN0Mk9wdE1vdmUuYyI6IDEsICJCZXN0M09wdE1vdmUuYyI6IDIsCiAgICAiQmVzdDRPcHRNb3ZlLmMiOiAzLCAiQmVzdDVPcHRNb3ZlLmMiOiA0LAogICAgIkJlc3RLT3B0TW92ZS5jIjogMSwgIkJlc3RTcGVjaWFsT3B0TW92ZS5jIjogMiwKICAgICJHYWluMjMuYyI6IDMsICJCcmlkZ2VHYWluLmMiOiAzLCAiUGF0Y2hDeWNsZXMuYyI6IDMsCn0KVE9QX0xFVkVMX0ZJTEVTID0gewogICAgIkJlc3QyT3B0TW92ZS5jIiwgIkJlc3QzT3B0TW92ZS5jIiwgIkJlc3Q0T3B0TW92ZS5jIiwKICAgICJCZXN0NU9wdE1vdmUuYyIsICJCZXN0S09wdE1vdmUuYyIsICJCZXN0U3BlY2lhbE9wdE1vdmUuYyIsCn0KCl9DQU5ESURBVEVfTE9PUCA9IHJlLmNvbXBpbGUoCiAgICByIig/bSleKFsgXHRdKilmb3JccypcKFxzKihOXHcrKVxzKj1ccyooXHcrKS0+Q2FuZGlkYXRlU2V0XHMqOyIKICAgIHIiXHMqXChccyooXHcrKVxzKj1ccypcMi0+VG9ccypcKVxzKjtccypcMlwrXCtccypcKVxzKlx7IgopCgoKZGVmIF9yZXBsYWNlX29uY2Uoc291cmNlOiBzdHIsIG9sZDogc3RyLCBuZXc6IHN0cikgLT4gc3RyOgogICAgY291bnQgPSBzb3VyY2UuY291bnQob2xkKQogICAgaWYgY291bnQgIT0gMToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiRXhwZWN0ZWQgb25lIHNvdXJjZSBhbmNob3IsIGZvdW5kIHtjb3VudH06IHtvbGRbOjcwXSFyfSIpCiAgICByZXR1cm4gc291cmNlLnJlcGxhY2Uob2xkLCBuZXcsIDEpCgoKZGVmIHBhdGNoX2ZpbmR0b3VyKHNvdXJjZTogc3RyKSAtPiBzdHI6CiAgICBzb3VyY2UgPSBfcmVwbGFjZV9vbmNlKHNvdXJjZSwgJyNpbmNsdWRlICJMS0guaCJcbicsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICcjaW5jbHVkZSAiTEtILmgiXG4jaW5jbHVkZSAiQmV0YVRvdXJTZWFyY2hIb29rLmgiXG4nKQogICAgc291cmNlID0gX3JlcGxhY2Vfb25jZShzb3VyY2UsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICIgICAgZG91YmxlIEVudHJ5VGltZSA9IEdldFRpbWUoKTtcblxuICAgIHQgPSBGaXJzdE5vZGU7IiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgIiAgICBkb3VibGUgRW50cnlUaW1lID0gR2V0VGltZSgpO1xuXG4iCiAgICAgICAgICAgICAgICAgICAgICAgICAgICIgICAgQlRfRXhwb3J0TmF0aXZlQ2FuZGlkYXRlcygpO1xuICAgIHQgPSBGaXJzdE5vZGU7IikKICAgIHNvdXJjZSA9IF9yZXBsYWNlX29uY2Uoc291cmNlLCAiICAgICAgICAgICAgUmVjb3JkQmV0dGVyVG91cigpO1xuIiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgIiAgICAgICAgICAgIFJlY29yZEJldHRlclRvdXIoKTtcbiAgICAgICAgICAgIEJUX1JlY29yZEJlc3QoKTtcbiIpCiAgICBzb3VyY2UgPSBfcmVwbGFjZV9vbmNlKHNvdXJjZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgIiAgICAgICAgfVxuICAgIH1cbiAgICBpZiAoQmFja2JvbmVUcmlhbHMgPiAwICYmIEJhY2tib25lVHJpYWxzIDwgTWF4VHJpYWxzKSB7IiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgIiAgICAgICAgfVxuICAgICAgICBCVF9BZnRlckZpcnN0VHJpYWwoKTtcbiIKICAgICAgICAgICAgICAgICAgICAgICAgICAgIiAgICB9XG4gICAgaWYgKEJhY2tib25lVHJpYWxzID4gMCAmJiBCYWNrYm9uZVRyaWFscyA8IE1heFRyaWFscykgeyIpCiAgICByZXR1cm4gc291cmNlCgoKZGVmIHBhdGNoX21vdmVfZmlsZShzb3VyY2U6IHN0ciwgZmlsZW5hbWU6IHN0cikgLT4gc3RyOgogICAgc291cmNlID0gX3JlcGxhY2Vfb25jZShzb3VyY2UsICcjaW5jbHVkZSAiTEtILmgiXG4nLAogICAgICAgICAgICAgICAgICAgICAgICAgICAnI2luY2x1ZGUgIkxLSC5oIlxuI2luY2x1ZGUgIkJldGFUb3VyVmlzaXQuaCJcbicpCgogICAgZGVmIGluc2VydF92aXNpdChtYXRjaDogcmUuTWF0Y2hbc3RyXSkgLT4gc3RyOgogICAgICAgIGluZGVudCwgX2NhbmRpZGF0ZSwgZnJvbV9uYW1lLCB0b19uYW1lID0gbWF0Y2guZ3JvdXBzKCkKICAgICAgICByZXR1cm4gbWF0Y2guZ3JvdXAoMCkgKyBmIlxue2luZGVudH0gICAgQlRfTm90ZUNhbmRpZGF0ZSh7ZnJvbV9uYW1lfSwge3RvX25hbWV9KTsiCgogICAgc291cmNlLCBjb3VudCA9IF9DQU5ESURBVEVfTE9PUC5zdWJuKGluc2VydF92aXNpdCwgc291cmNlKQogICAgaWYgY291bnQgIT0gTU9WRV9MT09QX0NPVU5UU1tmaWxlbmFtZV06CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIlVuZXhwZWN0ZWQgY2FuZGlkYXRlIGxvb3BzIGluIHtmaWxlbmFtZX06IHtjb3VudH0iKQogICAgaWYgZmlsZW5hbWUgaW4gVE9QX0xFVkVMX0ZJTEVTOgogICAgICAgIG9sZCA9ICJCVF9Ob3RlQ2FuZGlkYXRlKHQyLCB0Myk7IgogICAgICAgIHNvdXJjZSA9IF9yZXBsYWNlX29uY2Uoc291cmNlLCBvbGQsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBvbGQgKyAiXG4gICAgICAgIEJUX05vdGVUb3BMZXZlbEVudW1lcmF0ZWQodDIsIHQzKTsiKQogICAgICAgIGZpcnN0ID0gc291cmNlLmluZGV4KCJCVF9Ob3RlVG9wTGV2ZWxFbnVtZXJhdGVkKHQyLCB0Myk7IikKICAgICAgICBnYXRlID0gIiAgICAgICAgICAgIGNvbnRpbnVlO1xuIgogICAgICAgIGVuZCA9IHNvdXJjZS5maW5kKGdhdGUsIGZpcnN0KQogICAgICAgIGlmIGVuZCA8IDA6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJDYW5ub3QgZmluZCBmaXJzdC1zdGFnZSBnYWluIGdhdGUgaW4ge2ZpbGVuYW1lfSIpCiAgICAgICAgZW5kICs9IGxlbihnYXRlKQogICAgICAgIHNvdXJjZSA9IHNvdXJjZVs6ZW5kXSArICIgICAgICAgIEJUX05vdGVUb3BMZXZlbFBhc3ModDIsIHQzLCBHMSk7XG4iICsgc291cmNlW2VuZDpdCiAgICByZXR1cm4gc291cmNlCgoKZGVmIHBhdGNoX3N0b3JldG91cihzb3VyY2U6IHN0cikgLT4gc3RyOgogICAgc291cmNlID0gX3JlcGxhY2Vfb25jZShzb3VyY2UsICcjaW5jbHVkZSAiTEtILmgiXG4nLAogICAgICAgICAgICAgICAgICAgICAgICAgICAnI2luY2x1ZGUgIkxLSC5oIlxuI2luY2x1ZGUgIkJldGFUb3VyVmlzaXQuaCJcbicpCiAgICBzb3VyY2UgPSBfcmVwbGFjZV9vbmNlKHNvdXJjZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgIiAgICAgICAgICAgIEFjdGl2YXRlKHQpO1xuICAgICAgICAgICAgdC0+T2xkUHJlZCA9IHQtPlByZWQ7IiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgIiAgICAgICAgICAgIEFjdGl2YXRlKHQpO1xuIgogICAgICAgICAgICAgICAgICAgICAgICAgICAiICAgICAgICAgICAgQlRfTm90ZUFjY2VwdGVkRWRnZSh0KTtcbiIKICAgICAgICAgICAgICAgICAgICAgICAgICAgIiAgICAgICAgICAgIHQtPk9sZFByZWQgPSB0LT5QcmVkOyIpCiAgICByZXR1cm4gc291cmNlCgoKZGVmIF9jb21wYWN0KHRleHQ6IHN0cikgLT4gc3RyOgogICAgcmV0dXJuIHJlLnN1YihyIlxzKyIsICIiLCB0ZXh0KQoKCmRlZiBpbnNwZWN0X2NhbmRpZGF0ZV9yZW9yZGVyaW5nKHNyYzogUGF0aCkgLT4gZGljdDoKICAgICIiIlJlY29yZCwgd2l0aG91dCBwYXRjaGluZywgaG93IHVwc3RyZWFtIExLSCByZW9yZGVycyBjYW5kaWRhdGUgcm93cy4KCiAgICBGaW5kVG91ciBjYWxscyBBZGp1c3RDYW5kaWRhdGVTZXQgYWZ0ZXIgZWFjaCBiZXR0ZXIgdG91ciwgc28gYW4gYWN0aW9uJ3MKICAgIGluaXRpYWwgc2xvdCBpcyBub3Qgc3RhYmxlLiBUaGUgdjExIGhvb2sgZW5mb3JjZXMgdGhlIHNsb3QgYXMgYSBmbG9vciBhdAogICAgdHJpYWwgYm91bmRhcmllczsgdGhpcyBzdGF0aWMgY2hlY2sgZG9jdW1lbnRzIHdoaWNoIHVwc3RyZWFtIHJlb3JkZXJpbmcKICAgIHJ1bGUgdGhlIHNsb3QgZmxvb3IgY291bnRlcmFjdHMuIEl0IG5ldmVyIGZhaWxzIHRoZSBidWlsZC4KICAgICIiIgogICAgcmVwb3J0OiBkaWN0ID0ge30KICAgIHRleHRzID0ge30KICAgIGZvciBuYW1lIGluICgiRmluZFRvdXIuYyIsICJBZGp1c3RDYW5kaWRhdGVTZXQuYyIsICJSZXNldENhbmRpZGF0ZVNldC5jIik6CiAgICAgICAgcGF0aCA9IHNyYyAvIG5hbWUKICAgICAgICBpZiBwYXRoLmV4aXN0cygpOgogICAgICAgICAgICBkYXRhID0gcGF0aC5yZWFkX2J5dGVzKCkKICAgICAgICAgICAgdGV4dHNbbmFtZV0gPSBfY29tcGFjdChkYXRhLmRlY29kZSgidXRmLTgiLCBlcnJvcnM9InJlcGxhY2UiKSkKICAgICAgICAgICAgcmVwb3J0W25hbWUucmVwbGFjZSgiLmMiLCAiX3NoYTI1NiIpXSA9IGhhc2hsaWIuc2hhMjU2KGRhdGEpLmhleGRpZ2VzdCgpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgcmVwb3J0W25hbWUucmVwbGFjZSgiLmMiLCAiX3NoYTI1NiIpXSA9IE5vbmUKICAgIGZpbmR0b3VyID0gdGV4dHMuZ2V0KCJGaW5kVG91ci5jIiwgIiIpCiAgICBsb29wID0gZmluZHRvdXIuc3BsaXQoImZvcihUcmlhbD0xO1RyaWFsPD1NYXhUcmlhbHM7VHJpYWwrKykiLCAxKQogICAgcmVwb3J0WyJmaW5kdG91cl90cmlhbF9sb29wX2NhbGxzX2FkanVzdF9jYW5kaWRhdGVfc2V0Il0gPSAoCiAgICAgICAgbGVuKGxvb3ApID09IDIgYW5kICJBZGp1c3RDYW5kaWRhdGVTZXQoKTsiIGluIGxvb3BbMV0uc3BsaXQoIlJlc2V0Q2FuZGlkYXRlU2V0KCk7IiwgMSlbMF0pCiAgICBhZGp1c3QgPSB0ZXh0cy5nZXQoIkFkanVzdENhbmRpZGF0ZVNldC5jIiwgIiIpCiAgICByZXBvcnRbImFkanVzdF9tb3Zlc19lZGdlc19vZl90d29fYmVzdF90b3Vyc190b19mcm9udCJdID0gKAogICAgICAgICJJbkJlc3RUb3VyKEZyb20sVG8pJiZJbk5leHRCZXN0VG91cihGcm9tLFRvKSIgaW4gYWRqdXN0KQogICAgcmVwb3J0WyJhZGp1c3Rfc29ydHNfcm93c19ieV9hbHBoYV90aGVuX2Nvc3QiXSA9ICgKICAgICAgICAiVGVtcC5BbHBoYTxOTi0+QWxwaGEiIGluIGFkanVzdCkKICAgIHJlcG9ydFsicmVzZXRfc29ydHNfcm93c19ieV9hbHBoYV90aGVuX2Nvc3QiXSA9ICgKICAgICAgICAiVGVtcC5BbHBoYTxOTi0+QWxwaGEiIGluIHRleHRzLmdldCgiUmVzZXRDYW5kaWRhdGVTZXQuYyIsICIiKSkKICAgIHJldHVybiByZXBvcnQKCgpkZWYgYnVpbGQoc291cmNlX2RpcjogUGF0aCwgb3V0cHV0X2RpcjogUGF0aCwgam9iczogaW50ID0gNCkgLT4gUGF0aDoKICAgIHNvdXJjZV9kaXIsIG91dHB1dF9kaXIgPSBzb3VyY2VfZGlyLnJlc29sdmUoKSwgb3V0cHV0X2Rpci5yZXNvbHZlKCkKICAgIHNyYyA9IHNvdXJjZV9kaXIgLyAiU1JDIgogICAgaWYgbm90IChzb3VyY2VfZGlyIC8gIk1ha2VmaWxlIikuZXhpc3RzKCk6CiAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoZiJMS0ggc291cmNlIHRyZWUgbWlzc2luZzoge3NvdXJjZV9kaXJ9IikKICAgIG9yaWdpbmFscyA9IHt9CiAgICBmb3IgZmlsZW5hbWUsIGV4cGVjdGVkIGluIFVQU1RSRUFNX1NIQTI1Ni5pdGVtcygpOgogICAgICAgIGRhdGEgPSAoc3JjIC8gZmlsZW5hbWUpLnJlYWRfYnl0ZXMoKQogICAgICAgIGRpZ2VzdCA9IGhhc2hsaWIuc2hhMjU2KGRhdGEpLmhleGRpZ2VzdCgpCiAgICAgICAgaWYgZGlnZXN0ICE9IGV4cGVjdGVkOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiVW5leHBlY3RlZCB1cHN0cmVhbSBTSEEyNTYgZm9yIHtmaWxlbmFtZX06IHtkaWdlc3R9IikKICAgICAgICBvcmlnaW5hbHNbZmlsZW5hbWVdID0gZGF0YS5kZWNvZGUoInV0Zi04IikKICAgIGlmIG91dHB1dF9kaXIuZXhpc3RzKCk6CiAgICAgICAgcmFpc2UgRmlsZUV4aXN0c0Vycm9yKGYiUmVmdXNpbmcgdG8gb3ZlcndyaXRlIHtvdXRwdXRfZGlyfSIpCiAgICBzaHV0aWwuY29weXRyZWUoc291cmNlX2Rpciwgb3V0cHV0X2RpciwKICAgICAgICAgICAgICAgICAgICBpZ25vcmU9c2h1dGlsLmlnbm9yZV9wYXR0ZXJucygiT0JKIiwgIkxLSCIsICIqLm8iKSkKICAgIChvdXRwdXRfZGlyIC8gIlNSQyIgLyAiT0JKIikubWtkaXIoZXhpc3Rfb2s9VHJ1ZSkKICAgIChvdXRwdXRfZGlyIC8gIlNSQyIgLyAiRmluZFRvdXIuYyIpLndyaXRlX3RleHQoCiAgICAgICAgcGF0Y2hfZmluZHRvdXIob3JpZ2luYWxzWyJGaW5kVG91ci5jIl0pKQogICAgZm9yIGZpbGVuYW1lIGluIE1PVkVfTE9PUF9DT1VOVFM6CiAgICAgICAgKG91dHB1dF9kaXIgLyAiU1JDIiAvIGZpbGVuYW1lKS53cml0ZV90ZXh0KAogICAgICAgICAgICBwYXRjaF9tb3ZlX2ZpbGUob3JpZ2luYWxzW2ZpbGVuYW1lXSwgZmlsZW5hbWUpKQogICAgKG91dHB1dF9kaXIgLyAiU1JDIiAvICJTdG9yZVRvdXIuYyIpLndyaXRlX3RleHQoCiAgICAgICAgcGF0Y2hfc3RvcmV0b3VyKG9yaWdpbmFsc1siU3RvcmVUb3VyLmMiXSkpCiAgICBmb3IgaGVhZGVyIGluICgiQmV0YVRvdXJTZWFyY2hIb29rLmgiLCAiQmV0YVRvdXJWaXNpdC5oIik6CiAgICAgICAgc2h1dGlsLmNvcHkyKFBhdGgoX19maWxlX18pLndpdGhfbmFtZShoZWFkZXIpLCBvdXRwdXRfZGlyIC8gIlNSQyIgLyBoZWFkZXIpCiAgICByZXN1bHQgPSBzdWJwcm9jZXNzLnJ1bihbIm1ha2UiLCAiLWoiLCBzdHIoam9icyldLCBjd2Q9b3V0cHV0X2RpciwKICAgICAgICAgICAgICAgICAgICAgICAgICAgIHN0ZG91dD1zdWJwcm9jZXNzLlBJUEUsIHN0ZGVycj1zdWJwcm9jZXNzLlNURE9VVCwKICAgICAgICAgICAgICAgICAgICAgICAgICAgIHRleHQ9VHJ1ZSwgY2hlY2s9RmFsc2UpCiAgICBpZiByZXN1bHQucmV0dXJuY29kZToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJMS0ggaW5zdHJ1bWVudGVkIGJ1aWxkIGZhaWxlZDpcbntyZXN1bHQuc3Rkb3V0Wy04MDAwOl19IikKICAgIGJpbmFyeSA9IG91dHB1dF9kaXIgLyAiTEtIIgogICAgaWYgbm90IGJpbmFyeS5leGlzdHMoKToKICAgICAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcihmIk1pc3NpbmcgYnVpbHQgYmluYXJ5OiB7YmluYXJ5fSIpCiAgICBtYW5pZmVzdCA9IHsidXBzdHJlYW1fc2hhMjU2IjogVVBTVFJFQU1fU0hBMjU2LAogICAgICAgICAgICAgICAgInNlYXJjaF9ob29rX3NoYTI1NiI6IGhhc2hsaWIuc2hhMjU2KAogICAgICAgICAgICAgICAgICAgIFBhdGgoX19maWxlX18pLndpdGhfbmFtZSgiQmV0YVRvdXJTZWFyY2hIb29rLmgiKS5yZWFkX2J5dGVzKCkpLmhleGRpZ2VzdCgpLAogICAgICAgICAgICAgICAgInZpc2l0X2hlYWRlcl9zaGEyNTYiOiBoYXNobGliLnNoYTI1NigKICAgICAgICAgICAgICAgICAgICBQYXRoKF9fZmlsZV9fKS53aXRoX25hbWUoIkJldGFUb3VyVmlzaXQuaCIpLnJlYWRfYnl0ZXMoKSkuaGV4ZGlnZXN0KCksCiAgICAgICAgICAgICAgICAiYmluYXJ5X3NoYTI1NiI6IGhhc2hsaWIuc2hhMjU2KGJpbmFyeS5yZWFkX2J5dGVzKCkpLmhleGRpZ2VzdCgpLAogICAgICAgICAgICAgICAgImluc3RydW1lbnRlZF9tb3ZlX2xvb3BzIjogc3VtKE1PVkVfTE9PUF9DT1VOVFMudmFsdWVzKCkpLAogICAgICAgICAgICAgICAgImFjdGlvbl9zbG90X3J1bGUiOiAiZmxvb3IgZW5mb3JjZWQgYXQgZXZlcnkgY29tcGxldGVkLXRyaWFsIGJvdW5kYXJ5IiwKICAgICAgICAgICAgICAgICJ1cHN0cmVhbV9jYW5kaWRhdGVfcmVvcmRlcmluZyI6IGluc3BlY3RfY2FuZGlkYXRlX3Jlb3JkZXJpbmcoc3JjKX0KICAgIChvdXRwdXRfZGlyIC8gImJldGF0b3VyX2J1aWxkLmpzb24iKS53cml0ZV90ZXh0KGpzb24uZHVtcHMobWFuaWZlc3QsIGluZGVudD0yKSArICJcbiIpCiAgICByZXR1cm4gYmluYXJ5CgoKZGVmIG1haW4oKSAtPiBOb25lOgogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoZGVzY3JpcHRpb249X19kb2NfXykKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tc291cmNlLWRpciIsIHR5cGU9UGF0aCwgcmVxdWlyZWQ9VHJ1ZSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tb3V0cHV0LWRpciIsIHR5cGU9UGF0aCwgcmVxdWlyZWQ9VHJ1ZSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tam9icyIsIHR5cGU9aW50LCBkZWZhdWx0PTQpCiAgICBhcmdzID0gcGFyc2VyLnBhcnNlX2FyZ3MoKQogICAgcHJpbnQoYnVpbGQoYXJncy5zb3VyY2VfZGlyLCBhcmdzLm91dHB1dF9kaXIsIGFyZ3Muam9icykpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo=', 'v11_candidate/policy.py': '...', 'v11_candidate/pilot.py': '...', 'v11_candidate/run_cell.py': 'IiIidjExIENvbGFiIGV4ZWN1dGlvbiBjZWxsLiBDb21wbGV0ZWQgYXJtcyBhcmUgaW1tdXRhYmxlIGFuZCByZXN1bWFibGUuIiIiCgoKZGVmIGFybV9yZWNvcmRfcGF0aChpbnN0LCBhcm0pOgogICAgcmV0dXJuIE9VVFBVVF9ST09UIC8gImluc3RhbmNlcyIgLyBpbnN0WyJuYW1lIl0gLyBhcm0gLyAicmVzdWx0Lmpzb24iCgoKZGVmIHZhbGlkYXRlX3NhdmVkKGluc3QsIGFybSwgcmVjb3JkKToKICAgIGFzc2VydCByZWNvcmRbImNvbXBsZXRlIl0gYW5kIHJlY29yZFsiYXJtIl0gPT0gYXJtIGFuZCByZWNvcmRbImluc3RhbmNlIl0gPT0gaW5zdFsibmFtZSJdCiAgICBhc3NlcnQgcmVjb3JkWyJ4eV9zaGEyNTYiXSA9PSBpbnN0WyJ4eV9zaGEyNTYiXSBhbmQgcmVjb3JkWyJzZWVkcyJdID09IFNFRURTCiAgICBhc3NlcnQgcmVjb3JkWyJidWRnZXRfcyJdID09IGluc3RbImJ1ZGdldF9zIl0gYW5kIHJlY29yZFsiYnVpbGQiXSA9PSBCVUlMRF9JTkZPCiAgICBhc3NlcnQgcmVjb3JkWyJjb2RlX3NoYTI1NiJdID09IENPREVfU0hBMjU2CiAgICBhc3NlcnQgcmVjb3JkWyJmcmFjdGlvbiJdID09IFNQQVJTRV9GUkFDVElPTiBhbmQgcmVjb3JkWyJncmFwaF9rIl0gPT0gR1JBUEhfSwogICAgYXNzZXJ0IGxlbihyZWNvcmRbInNlZWRfY29zdHMiXSkgPT0gbGVuKFNFRURTKQogICAgYXNzZXJ0IG1pbihyZWNvcmRbInNlZWRfY29zdHMiXSkgPT0gcmVjb3JkWyJiZXN0X2Nvc3QiXQogICAgaWYgYXJtIGluICgibmF0aXZlX3BvcDYiLCAibmF0aXZlX3BvcDciKToKICAgICAgICBhc3NlcnQgc2V0KHJlY29yZFsiZmluYWxfdG91cl9zb3VyY2VzIl0pID09IHtzdHIoc2VlZCkgZm9yIHNlZWQgaW4gU0VFRFN9CiAgICBlbHNlOgogICAgICAgIGFzc2VydCBsZW4ocmVjb3JkWyJmaXJzdF90b3VyX2hhc2hlcyJdKSA9PSBsZW4oU0VFRFMpCiAgICAgICAgYXNzZXJ0IGxlbihyZWNvcmRbImxpdmVfY2FuZGlkYXRlX2hhc2hlcyJdKSA9PSBsZW4oU0VFRFMpCiAgICAgICAgYXNzZXJ0IGxlbihyZWNvcmRbIm5hdGl2ZV9jYW5kaWRhdGVfaGFzaGVzIl0pID09IGxlbihTRUVEUykKICAgICAgICBhdWRpdCA9IHJlY29yZFsicG9saWN5X2F1ZGl0Il0KICAgICAgICBhc3NlcnQgbGVuKGF1ZGl0WyJnYXRlX3NoYTI1Nl9wZXJfc2VlZCJdKSA9PSBsZW4oU0VFRFMpCiAgICAgICAgYXNzZXJ0IGF1ZGl0WyJzZWVkX2lkcyJdID09IFNFRURTCiAgICAgICAgYXNzZXJ0IGF1ZGl0WyJwb2xpY3lfdmVyc2lvbiJdID09IFBPTElDWV9WRVJTSU9OCiAgICAgICAgYXNzZXJ0IGF1ZGl0WyJkZWNpc2lvbl9iYXNpcyJdIGluICgiZnJlc2hfaW5mZXJlbmNlIiwgImNhY2hlZF9pbmZlcmVuY2UiKQogICAgICAgIGlmIGFybSA9PSAic2hhbSI6CiAgICAgICAgICAgIGFzc2VydCBhdWRpdFsic2VsZWN0ZWRfYWN0aW9uc19wZXJfc2VlZCJdID09IFswXSAqIGxlbihTRUVEUykKICAgICAgICBlbHNlOgogICAgICAgICAgICBhc3NlcnQgYXVkaXRbInNlbGVjdGVkX2FjdGlvbnNfcGVyX3NlZWQiXSA9PSBhdWRpdFsiZ3VpZGVkX2FjdGlvbnNfcGVyX3NlZWQiXQogICAgICAgIGFzc2VydCBhdWRpdFsic2VsZWN0ZWRfdW5pcXVlX3VuZGlyZWN0ZWRfZWRnZXNfcGVyX3NlZWQiXSA9PSBhdWRpdFsic2VsZWN0ZWRfYWN0aW9uc19wZXJfc2VlZCJdCiAgICByZXR1cm4gcmVjb3JkCgoKZGVmIHNhdmVfcHJvZ3Jlc3Moc3RhdGU9InJ1bm5pbmciLCBsYXN0PU5vbmUpOgogICAgY29tcGxldGVkID0gc3VtKGFybV9yZWNvcmRfcGF0aChpbnN0LCBhcm0pLmV4aXN0cygpIGZvciBpbnN0IGluIElOU1RBTkNFUyBmb3IgYXJtIGluIEFSTVMpCiAgICB3cml0ZV9qc29uX2F0b21pYyhPVVRQVVRfUk9PVCAvICJwcm9ncmVzcy5qc29uIiwKICAgICAgICAgICAgICAgICAgICAgIHsic3RhdGUiOiBzdGF0ZSwgImNvbXBsZXRlZF9hcm1zIjogY29tcGxldGVkLAogICAgICAgICAgICAgICAgICAgICAgICJ0b3RhbF9hcm1zIjogbGVuKElOU1RBTkNFUykgKiBsZW4oQVJNUyksICJsYXN0IjogbGFzdCwKICAgICAgICAgICAgICAgICAgICAgICAidXBkYXRlZF91dGMiOiB0aW1lLnN0cmZ0aW1lKCIlWS0lbS0lZFQlSDolTTolU1oiLCB0aW1lLmdtdGltZSgpKX0pCgoKbmV3X2FybXMgPSAwCnNhdmVfcHJvZ3Jlc3MoKQpmb3IgaW5zdCBpbiBJTlNUQU5DRVM6CiAgICBpbnN0YW5jZV9kaXIgPSBPVVRQVVRfUk9PVCAvICJpbnN0YW5jZXMiIC8gaW5zdFsibmFtZSJdCiAgICBpbnN0YW5jZV9kaXIubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgY29vcmRfcGF0aCA9IGluc3RhbmNlX2RpciAvICJ4eS5ucHoiCiAgICBpZiBub3QgY29vcmRfcGF0aC5leGlzdHMoKToKICAgICAgICB0ZW1wID0gaW5zdGFuY2VfZGlyIC8gInh5LnRtcC5ucHoiCiAgICAgICAgbnAuc2F2ZXpfY29tcHJlc3NlZCh0ZW1wLCB4eT1pbnN0WyJ4eSJdKQogICAgICAgIG9zLnJlcGxhY2UodGVtcCwgY29vcmRfcGF0aCkKICAgIHdpdGggbnAubG9hZChjb29yZF9wYXRoLCBhbGxvd19waWNrbGU9RmFsc2UpIGFzIHNhdmVkX3h5OgogICAgICAgIGFzc2VydCBoYXNobGliLnNoYTI1NihucC5hc2NvbnRpZ3VvdXNhcnJheShzYXZlZF94eVsieHkiXSkudG9ieXRlcygpKS5oZXhkaWdlc3QoKSA9PSBpbnN0WyJ4eV9zaGEyNTYiXQogICAgbG9jYWxfcHJvYmxlbSA9IFdPUktfUk9PVCAvICJwcm9ibGVtcyIgLyAoaW5zdFsibmFtZSJdICsgIi50c3AiKQogICAgbG9jYWxfcHJvYmxlbS5wYXJlbnQubWtkaXIoZXhpc3Rfb2s9VHJ1ZSkKICAgIFQud3JpdGVfdHNwKGxvY2FsX3Byb2JsZW0sIGluc3RbInh5Il0pCiAgICBmb3IgYXJtIGluIGFybV9vcmRlcihpbnN0WyJuYW1lIl0pOgogICAgICAgIHJlY29yZF9wYXRoID0gYXJtX3JlY29yZF9wYXRoKGluc3QsIGFybSkKICAgICAgICBpZiByZWNvcmRfcGF0aC5leGlzdHMoKToKICAgICAgICAgICAgdmFsaWRhdGVfc2F2ZWQoaW5zdCwgYXJtLCBqc29uLmxvYWRzKHJlY29yZF9wYXRoLnJlYWRfdGV4dCgpKSkKICAgICAgICAgICAgcHJpbnQoIkFUTEE6IiwgaW5zdFsibmFtZSJdLCBhcm0sIGZsdXNoPVRydWUpCiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgaWYgTUFYX05FV19BUk1TIGlzIG5vdCBOb25lIGFuZCBuZXdfYXJtcyA+PSBNQVhfTkVXX0FSTVM6CiAgICAgICAgICAgIGJyZWFrCiAgICAgICAgYXR0ZW1wdF9pZCA9IHV1aWQudXVpZDQoKS5oZXgKICAgICAgICBsb2NhbF9ydW4gPSBXT1JLX1JPT1QgLyAicnVucyIgLyBpbnN0WyJuYW1lIl0gLyBhcm0gLyBhdHRlbXB0X2lkCiAgICAgICAgbG9jYWxfcnVuLnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICAgICAgdHJ5OgogICAgICAgICAgICBwcmludCgiQkHFnkxBOiIsIGluc3RbIm5hbWUiXSwgYXJtLCAiYsO8dMOnZSIsIGluc3RbImJ1ZGdldF9zIl0sICJzbiIsIGZsdXNoPVRydWUpCiAgICAgICAgICAgIGlmIGFybS5zdGFydHN3aXRoKCJuYXRpdmVfIik6CiAgICAgICAgICAgICAgICByYXcgPSBydW5fbmF0aXZlX2FybShhcm0sIE5BVElWRV9MS0gsIGxvY2FsX3Byb2JsZW0sIGluc3RbInh5Il0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBsb2NhbF9ydW4sIFNFRURTLCBpbnN0WyJidWRnZXRfcyJdLCBUKQogICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgIyBBcm1zIHdob3NlIG1vZGVsIGlucHV0IGlzIGJpdHdpc2UgaWRlbnRpY2FsIGRlY2lkZSBmcm9tIHRoZQogICAgICAgICAgICAgICAgIyBmaXJzdCBzdG9yZWQgbW9kZWwgb3V0cHV0OyBldmVyeSBhcm0gc3RpbGwgcnVucyBpbmZlcmVuY2UuCiAgICAgICAgICAgICAgICByYXcgPSBydW5fZ3VpZGVkX2FybShhcm0sIFBBVENIRURfTEtILCBsb2NhbF9wcm9ibGVtLCBpbnN0WyJ4eSJdLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgbG9jYWxfcnVuLCBTRUVEUywgaW5zdFsiYnVkZ2V0X3MiXSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIE1PREVMLCBERVZJQ0UsIFQsIE0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBmcmFjdGlvbj1TUEFSU0VfRlJBQ1RJT04sIGdyYXBoX2s9R1JBUEhfSywKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGRlY2lzaW9uX2NhY2hlX2Rpcj1pbnN0YW5jZV9kaXIgLyAiZGVjaXNpb25fY2FjaGUiLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgbW9kZWxfaWRlbnRpdHk9Q0hFQ0tQT0lOVF9TSEEyNTYpCiAgICAgICAgICAgIGF1ZGl0X3JlbCA9IGYiaW5zdGFuY2VzL3tpbnN0WyduYW1lJ119L3thcm19L3thdHRlbXB0X2lkfSIKICAgICAgICAgICAgYXVkaXRfZGVzdCA9IE9VVFBVVF9ST09UIC8gYXVkaXRfcmVsCiAgICAgICAgICAgIHNodXRpbC5jb3B5dHJlZShsb2NhbF9ydW4sIGF1ZGl0X2Rlc3QpCiAgICAgICAgICAgIHJlY29yZCA9IHsic2NoZW1hIjogMSwgImNvbXBsZXRlIjogVHJ1ZSwgImluc3RhbmNlIjogaW5zdFsibmFtZSJdLAogICAgICAgICAgICAgICAgICAgICAgImdyb3VwIjogaW5zdFsiZ3JvdXAiXSwgIm4iOiBpbnN0WyJuIl0sCiAgICAgICAgICAgICAgICAgICAgICAieHlfc2hhMjU2IjogaW5zdFsieHlfc2hhMjU2Il0sICJhcm0iOiBhcm0sCiAgICAgICAgICAgICAgICAgICAgICAic2VlZHMiOiBTRUVEUywgImJ1ZGdldF9zIjogaW5zdFsiYnVkZ2V0X3MiXSwKICAgICAgICAgICAgICAgICAgICAgICJzZWVkX2Nvc3RzIjogcmF3WyJzZWVkX2Nvc3RzIl0sICJiZXN0X2Nvc3QiOiByYXdbImJlc3RfY29zdCJdLAogICAgICAgICAgICAgICAgICAgICAgIndhbGxfZWxhcHNlZF9zIjogcmF3WyJ3YWxsX2VsYXBzZWRfcyJdLAogICAgICAgICAgICAgICAgICAgICAgInN0YXJ0X2Vwb2NoX3MiOiByYXdbInN0YXJ0X2Vwb2NoX3MiXSwKICAgICAgICAgICAgICAgICAgICAgICJkZWFkbGluZV9lcG9jaF9zIjogcmF3WyJkZWFkbGluZV9lcG9jaF9zIl0gaWYgImRlYWRsaW5lX2Vwb2NoX3MiIGluIHJhdyBlbHNlIHJhd1sic3RhcnRfZXBvY2hfcyJdICsgaW5zdFsiYnVkZ2V0X3MiXSwKICAgICAgICAgICAgICAgICAgICAgICJmaXJzdF9jb3N0cyI6IHJhdy5nZXQoImZpcnN0X2Nvc3RzIiksCiAgICAgICAgICAgICAgICAgICAgICAiZmlyc3RfdG91cl9oYXNoZXMiOiByYXcuZ2V0KCJmaXJzdF90b3VyX2hhc2hlcyIpLAogICAgICAgICAgICAgICAgICAgICAgImxpdmVfY2FuZGlkYXRlX2hhc2hlcyI6IHJhdy5nZXQoImxpdmVfY2FuZGlkYXRlX2hhc2hlcyIpLAogICAgICAgICAgICAgICAgICAgICAgIm5hdGl2ZV9jYW5kaWRhdGVfaGFzaGVzIjogcmF3LmdldCgibmF0aXZlX2NhbmRpZGF0ZV9oYXNoZXMiKSwKICAgICAgICAgICAgICAgICAgICAgICJwb2xpY3lfYXVkaXQiOiByYXcuZ2V0KCJwb2xpY3lfYXVkaXQiKSwKICAgICAgICAgICAgICAgICAgICAgICJndWlkZWRfaW1wcm92ZW1lbnRfZXZlbnRzIjogcmF3LmdldCgiZ3VpZGVkX2ltcHJvdmVtZW50X2V2ZW50cyIpLAogICAgICAgICAgICAgICAgICAgICAgIm5hdGl2ZV9pbXByb3ZlbWVudF9ldmVudHMiOiByYXcuZ2V0KCJuYXRpdmVfaW1wcm92ZW1lbnRfZXZlbnRzIiksCiAgICAgICAgICAgICAgICAgICAgICAiZmluYWxfdG91cl9zb3VyY2VzIjogcmF3LmdldCgiZmluYWxfdG91cl9zb3VyY2VzIiksCiAgICAgICAgICAgICAgICAgICAgICAiZXZlbnRfZmxhZ3MiOiByYXcuZ2V0KCJldmVudF9mbGFncyIpLAogICAgICAgICAgICAgICAgICAgICAgImRlY2lzaW9uX2NhY2hlX2ZpbGVzX3dyaXR0ZW4iOiByYXcuZ2V0KCJkZWNpc2lvbl9jYWNoZV9maWxlc193cml0dGVuIiksCiAgICAgICAgICAgICAgICAgICAgICAiZnJhY3Rpb24iOiBTUEFSU0VfRlJBQ1RJT04sICJncmFwaF9rIjogR1JBUEhfSywKICAgICAgICAgICAgICAgICAgICAgICJidWlsZCI6IEJVSUxEX0lORk8sICJjb2RlX3NoYTI1NiI6IENPREVfU0hBMjU2LAogICAgICAgICAgICAgICAgICAgICAgImhhcmR3YXJlIjogSEFSRFdBUkUsICJhdWRpdF9yZWxwYXRoIjogYXVkaXRfcmVsLAogICAgICAgICAgICAgICAgICAgICAgImF0dGVtcHRfaWQiOiBhdHRlbXB0X2lkLAogICAgICAgICAgICAgICAgICAgICAgImNvbXBsZXRlZF91dGMiOiB0aW1lLnN0cmZ0aW1lKCIlWS0lbS0lZFQlSDolTTolU1oiLCB0aW1lLmdtdGltZSgpKX0KICAgICAgICAgICAgdmFsaWRhdGVfc2F2ZWQoaW5zdCwgYXJtLCByZWNvcmQpCiAgICAgICAgICAgIHdyaXRlX2pzb25fYXRvbWljKHJlY29yZF9wYXRoLCByZWNvcmQpCiAgICAgICAgICAgIG5ld19hcm1zICs9IDEKICAgICAgICAgICAgc2F2ZV9wcm9ncmVzcyhsYXN0PWluc3RbIm5hbWUiXSArICIvIiArIGFybSkKICAgICAgICAgICAgcHJpbnQoIkLEsFRUxLA6IiwgaW5zdFsibmFtZSJdLCBhcm0sICJlbiBpeWkiLCByYXdbImJlc3RfY29zdCJdLCBmbHVzaD1UcnVlKQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZXhjOgogICAgICAgICAgICB3cml0ZV9qc29uX2F0b21pYyhPVVRQVVRfUk9PVCAvICJlcnJvcnMiIC8KICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgKGluc3RbIm5hbWUiXSArICJfIiArIGFybSArICJfIiArIGF0dGVtcHRfaWQgKyAiLmpzb24iKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgeyJpbnN0YW5jZSI6IGluc3RbIm5hbWUiXSwgImFybSI6IGFybSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICJlcnJvciI6IHJlcHIoZXhjKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICJ1dGMiOiB0aW1lLnN0cmZ0aW1lKCIlWS0lbS0lZFQlSDolTTolU1oiLCB0aW1lLmdtdGltZSgpKX0pCiAgICAgICAgICAgIHNhdmVfcHJvZ3Jlc3Moc3RhdGU9ImVycm9yIiwgbGFzdD1pbnN0WyJuYW1lIl0gKyAiLyIgKyBhcm0pCiAgICAgICAgICAgIHJhaXNlCiAgICBpZiBNQVhfTkVXX0FSTVMgaXMgbm90IE5vbmUgYW5kIG5ld19hcm1zID49IE1BWF9ORVdfQVJNUzoKICAgICAgICBicmVhawoKY29tcGxldGVkX2NvdW50ID0gc3VtKGFybV9yZWNvcmRfcGF0aChpbnN0LCBhcm0pLmV4aXN0cygpIGZvciBpbnN0IGluIElOU1RBTkNFUyBmb3IgYXJtIGluIEFSTVMpCnNhdmVfcHJvZ3Jlc3Moc3RhdGU9ImNvbXBsZXRlIiBpZiBjb21wbGV0ZWRfY291bnQgPT0gbGVuKElOU1RBTkNFUykgKiBsZW4oQVJNUykgZWxzZSAicGFydGlhbCIpCnByaW50KGYiVGFtYW1sYW5hbiBrb2w6IHtjb21wbGV0ZWRfY291bnR9L3tsZW4oSU5TVEFOQ0VTKSAqIGxlbihBUk1TKX0uICIKICAgICAgZiJCdSBvdHVydW1kYWtpIHllbmkga29sOiB7bmV3X2FybXN9IikK', 'v11_candidate/analysis_cell.py': '...', 'v11_candidate/build_colab_notebook.py': 'IiIiQnVpbGQgdGhlIHNlbGYtY29udGFpbmVkLCByZXN1bWFibGUgdjExcjIgUE9QNi1yZXNpZHVhbCBtZWNoYW5pc20gbm90ZWJvb2suCgpUaGUgcnVuLXYxMSBub3RlYm9vayAoMDVfLi4uKSBpcyBrZXB0IHVuY2hhbmdlZCBhcyB0aGUgcmVjb3JkIG9mIHRoZSAyLXZDUFUgcnVuLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBiYXNlNjQKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGltcG9ydGxpYi51dGlsCmltcG9ydCBqc29uCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKCkhFUkUgPSBQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50CkVYUEVSSU1FTlRTID0gSEVSRS5wYXJlbnQKUkVQTyA9IEVYUEVSSU1FTlRTLnBhcmVudApPVVRQVVQgPSBSRVBPIC8gIm5vdGVib29rcyIgLyAiMDZfQmV0YVRvdXJfdjExcjJfUE9QNl9jYW5kaWRhdGVfbWVjaGFuaXNtLmlweW5iIgpWOV9CVUlMREVSID0gRVhQRVJJTUVOVFMgLyAidjlfbGl2ZSIgLyAiYnVpbGRfY29sYWJfbm90ZWJvb2sucHkiCnNwZWMgPSBpbXBvcnRsaWIudXRpbC5zcGVjX2Zyb21fZmlsZV9sb2NhdGlvbigiYmV0YXRvdXJfdjlfYnVpbGRlciIsIFY5X0JVSUxERVIpCmlmIHNwZWMgaXMgTm9uZSBvciBzcGVjLmxvYWRlciBpcyBOb25lOgogICAgcmFpc2UgUnVudGltZUVycm9yKCJ2OSBub3RlYm9vayBidWlsZGVyIG1pc3NpbmciKQp2OSA9IGltcG9ydGxpYi51dGlsLm1vZHVsZV9mcm9tX3NwZWMoc3BlYykKc3BlYy5sb2FkZXIuZXhlY19tb2R1bGUodjkpCgoKZGVmIHJlcGxhY2Vfb25jZShzb3VyY2U6IHN0ciwgb2xkOiBzdHIsIG5ldzogc3RyKSAtPiBzdHI6CiAgICBpZiBzb3VyY2UuY291bnQob2xkKSAhPSAxOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJleHBlY3RlZCBleGFjdGx5IG9uZSBub3RlYm9vayBhbmNob3I6IHtvbGRbOjcwXSFyfSIpCiAgICByZXR1cm4gc291cmNlLnJlcGxhY2Uob2xkLCBuZXcsIDEpCgoKZGVmIHNldF9jZWxsKG5vdGVib29rOiBkaWN0LCBpbmRleDogaW50LCBzb3VyY2U6IHN0cikgLT4gTm9uZToKICAgIGNlbGwgPSBub3RlYm9va1siY2VsbHMiXVtpbmRleF0KICAgIGlmIGNlbGxbImNlbGxfdHlwZSJdID09ICJjb2RlIjoKICAgICAgICBjb21waWxlKHNvdXJjZSwgZiI8djExIGNlbGwge2luZGV4fT4iLCAiZXhlYyIpCiAgICBjZWxsWyJzb3VyY2UiXSA9IHNvdXJjZS5zcGxpdGxpbmVzKGtlZXBlbmRzPVRydWUpCiAgICBjZWxsWyJpZCJdID0gZiJiZXRhdG91cl92MTFfe2luZGV4OjAyZH1fe2hhc2hsaWIuc2hhMjU2KHNvdXJjZS5lbmNvZGUoKSkuaGV4ZGlnZXN0KClbOjEwXX0iCgoKZGVmIG1ha2Vfbm90ZWJvb2soKSAtPiBkaWN0OgogICAgbm90ZWJvb2sgPSB2OS5tYWtlX25vdGVib29rKCkKICAgIG9sZCA9IGxhbWJkYSBpbmRleDogIiIuam9pbihub3RlYm9va1siY2VsbHMiXVtpbmRleF1bInNvdXJjZSJdKQoKICAgIHNldF9jZWxsKG5vdGVib29rLCAwLCAiIiIjIEJldGFUb3VyIHYxMXIyIOKAlCBQT1A2IGFkYXkga2VuYXIgbWVrYW5pem1hIHBpbG90dSAoZMO8emVsdGlsbWnFnyB0ZWtyYXIpCgpCdSBkZW5leSAqKmVza2kgdjgvdjkgZ2VsacWfdGlybWUgw7ZybmVrbGVyaW5pbiBkw7ZydCAxMGsgw7ZybmXEn2luaSoqIGt1bGxhbsSxcjsgYmHEn8SxbXPEsXogZG/En3J1bGFtYSBkZcSfaWxkaXIuIFBPUE1VU0lDLTYvTEtIJ25pbiBheW7EsSBpbGsgZGVuZW1lc2luZGVuIHNvbnJhIEJldGFUb3VyLCB0dXIgc8Sxa2zEscSfxLEgdmV5YSByYXN0Z2VsZSBrb250cm9sLCBheW7EsSBrYXluYWsgZMO8xJ/DvG1sZXJlIHZlIGF5bsSxIHNhecSxZGEgeWVuaSB5w7Zuc8O8eiBhZGF5IGtlbmFyYSBtw7xkYWhhbGUgZWRlci4gU2hhbSBheW7EsSBtb2RlbCDDp8Sxa2FyxLFtxLFuxLEgeWFwYXIgdmUgYXJhbWF5xLEgZGXEn2nFn3Rpcm1lei4gTmF0aXZlIFBPUDYgdmUgUE9QNyBwcmF0aWsga8SxeWFzbGFyZMSxci4gQsO8dMO8biBrb2wgYsO8dMOnZWxlcmluZSBrdXJ1bHVtLCBtb2RlbCDDp8Sxa2FyxLFtxLEsIGJla2xlbWUgdmUgTEtIIGFyYW1hc8SxIGRhaGlsZGlyLgoKKip2MTEnZSBnw7ZyZSBkZcSfacWfaWtsaWtsZXI6KiogKDEpIExLSCBoZXIgZGFoYSBpeWkgdHVyZGFuIHNvbnJhIGFkYXkgc2F0xLFybGFyxLFuxLEgeWVuaWRlbiBkw7x6ZW5sZWRpxJ9pIGnDp2luIGVrbGVuZW4ga2VuYXLEsW4gMy4gc8SxcmFzxLEgYXJ0xLFrIGhlciBkZW5lbWUgc29udW5kYSBiaXIgKnRhYmFuKiBvbGFyYWsga29ydW51ciB2ZSBrYcOnIGtleiBnZXJpIGFsxLFuZMSxxJ/EsSBzYXnEsWzEsXIuICgyKSBBeW7EsSBtb2RlbCBnaXJkaXNpbmkgYWxhbiBrb2xsYXIgaWxrIHNha2xhbmFuIG1vZGVsIMOnxLFrdMSxc8SxeWxhIGthcmFyIHZlcmlyOyBoZXIga29sIHlpbmUga2VuZGkgw6fEsWthcsSxbcSxbsSxIHlhcGFyIHZlIHPDvHJlc2luaSDDtmRlci4gKDMpIE5hdGl2ZSBrb2xsYXLEsW4gaXpsZW1lc2kgemFtYW4gc8SxbsSxcsSxIGnDp2luZGUgdHVyIGF5csSxxZ90xLFybWF6LiAoNCkgS2VzaW50aWRlIHlhcsSxbSBrYWxtxLHFnyB0dXIgdmUgb2xheSBrYXnEsXRsYXLEsSBnw7x2ZW5sZSBlbGUgYWzEsW7EsXIuIMSwbGsgdjExIMOnYWzEscWfbWFzxLEgVDQgdmUgMiB2Q1BVIMO8emVyaW5kZSBrb8WfdHXEn3UgacOnaW4gYnUgdGVrcmFyIGVuIGF6IDEyIHZDUFUgaXN0ZXIuCgoqKsOHYWzEscWfdMSxcm1hOioqIENvbGFiJ2RhICoqQTEwMCAodmV5YSBMNCkgR1BVIHZlIFnDvGtzZWsgUkFNKiogc2XDp2luOyBkZWZ0ZXIgMTInZGVuIGF6IHZDUFUgZ8O2csO8cnNlIGR1cnVyLiBIw7xjcmVsZXJpIHPEsXJheWxhIMOnYWzEscWfdMSxcsSxbi4gVGFtYW1sYW5hbiBrb2xsYXIgYE15RHJpdmUvQmV0YVRvdXIvdjExcjJfcG9wNl9jYW5kaWRhdGVfbWVjaGFuaXNtYCBpw6dpbmRlIGRvxJ9ydWxhbmFyYWsgYXRsYW7EsXI7IGtlc2lsZW4ga29sIHllbmlkZW4gw6dhbMSxxZ/EsXIuIFNvbnXDpyBaSVAnaSBzb251bmRhIHRhcmF5xLFjxLF5YSBvdG9tYXRpayBpbmVyLCB0YW0gZGVuZXRpbSBaSVAnaSBEcml2ZSdkYSBrYWzEsXIuCgoqKlRhaG1pbmkgc8O8cmU6Kioga3VydWx1bSB2ZSBMS0ggZGVybGVtZXNpIOKJiCAz4oCTNSBkazsgxLFzxLFubWEgdmUgZG/En3J1bGFtYSDiiYggMSBkazsgZMO2cnQgw7ZybmVrIMOXIGFsdMSxIGtvbCDDlyAxMjAgc24gPSAqKjQ4IGRrKiogw6fDtnrDvGPDvDsga2F5xLF0IHZlIGFyxZ9pdiDiiYggMuKAkzQgZGsuIFRvcGxhbSDiiYggNTXigJM2MCBkay4KIiIiKQoKICAgIGNvbmZpZyA9IG9sZCgxKQogICAgY29uZmlnID0gcmVwbGFjZV9vbmNlKGNvbmZpZywgJ0VYUEVSSU1FTlRfTkFNRSA9ICJ2OV9saXZlX3BpbG90IicsCiAgICAgICAgICAgICAgICAgICAgICAgICAgJ0VYUEVSSU1FTlRfTkFNRSA9ICJ2MTFyMl9wb3A2X2NhbmRpZGF0ZV9tZWNoYW5pc20iJykKICAgIGNvbmZpZyA9IHJlcGxhY2Vfb25jZShjb25maWcsCiAgICAgICAgJ0FSTVMgPSBbIm5hdGl2ZV9wb3A1IiwgIm5hdGl2ZV9wb3A2IiwgInNoYW0iLCAiYmV0YV9wcm9tb3RlIiwgImJldGFfc3BhcnNlIiwgImZyZXF1ZW5jeV9zYW1lX2dhdGUiXScsCiAgICAgICAgJ0FSTVMgPSBbIm5hdGl2ZV9wb3A2IiwgIm5hdGl2ZV9wb3A3IiwgInNoYW0iLCAicmFuZG9tX2FkZCIsICJmcmVxdWVuY3lfYWRkIiwgImJldGFfYWRkIl0nKQogICAgY29uZmlnID0gcmVwbGFjZV9vbmNlKGNvbmZpZywKICAgICAgICAnICAgIFsidW5pMjBrIiwgMjAwMDAsIDIsICJ1bmlmb3JtIiwgNDMwOSwgMzAwLjBdLFxuICAgIFsiY2x1MjBrIiwgMjAwMDAsIDIsICJjbHVzdGVyZWQiLCA0NDEwLCAzMDAuMF0sXG4nLCAnJykKICAgIGNvbmZpZyA9IHJlcGxhY2Vfb25jZShjb25maWcsICdTUEFSU0VfRlJBQ1RJT04gPSAwLjI1JywKICAgICAgICAgICAgICAgICAgICAgICAgICAnU1BBUlNFX0ZSQUNUSU9OID0gMC4yNVxuR1JBUEhfSyA9IDE2JykKICAgIGNvbmZpZyA9IHJlcGxhY2Vfb25jZShjb25maWcsICdhc3NlcnQgbGVuKFNFRURTKSA9PSA2IGFuZCBsZW4oRVhQRUNURURfWFlfU0hBMjU2KSA9PSA4JywKICAgICAgICAnRVhQRUNURURfWFlfU0hBMjU2ID0ge25hbWU6IGRpZ2VzdCBmb3IgbmFtZSwgZGlnZXN0IGluIEVYUEVDVEVEX1hZX1NIQTI1Ni5pdGVtcygpXG4nCiAgICAgICAgJyAgICAgICAgICAgICAgICAgICAgICBpZiAiMTBrIiBpbiBuYW1lfVxuJwogICAgICAgICcjIFRpbWUtdG8tdGFyZ2V0IGNvc3RzOiBydW4tdjEwIG5hdGl2ZSBQT1BNVVNJQy02IGJlc3Qgb2Ygc2l4IChBMTAwLCAxMiB2Q1BVKS5cbicKICAgICAgICAnVEFSR0VUX0NPU1RTID0geyJhdWdtZW50X2NsdTEwa18wMDAiOiAzMzc2MDE5OSxcbicKICAgICAgICAnICAgICAgICAgICAgICAgICJhdWdtZW50X2NsdTEwa18wMDEiOiAxNjIwMTE4NixcbicKICAgICAgICAnICAgICAgICAgICAgICAgICJhdWdtZW50X3VuaTEwa18wMDAiOiA3MjA4Nzc4MSxcbicKICAgICAgICAnICAgICAgICAgICAgICAgICJhdWdtZW50X3VuaTEwa18wMDEiOiA3MTY0ODU3MH1cbicKICAgICAgICAnYXNzZXJ0IGxlbihTRUVEUykgPT0gNiBhbmQgbGVuKEVYUEVDVEVEX1hZX1NIQTI1NikgPT0gNCBhbmQgbGVuKFRBUkdFVF9DT1NUUykgPT0gNCcpCiAgICBzZXRfY2VsbChub3RlYm9vaywgMSwgY29uZmlnKQoKICAgIHNldHVwID0gb2xkKDIpCiAgICBzZXR1cCA9IHJlcGxhY2Vfb25jZShzZXR1cCwgJ2JldGF0b3VyX3Y5X2xpdmVfd29yaycsICdiZXRhdG91cl92MTFyMl9jYW5kaWRhdGVfd29yaycpCiAgICBzZXR1cCA9IHJlcGxhY2Vfb25jZShzZXR1cCwKICAgICAgICAnInB5dGhvbiI6IHBsYXRmb3JtLnB5dGhvbl92ZXJzaW9uKCksICJ0b3JjaCI6IHRvcmNoLl9fdmVyc2lvbl9ffScsCiAgICAgICAgJyJweXRob24iOiBwbGF0Zm9ybS5weXRob25fdmVyc2lvbigpLCAidG9yY2giOiB0b3JjaC5fX3ZlcnNpb25fXyxcbicKICAgICAgICAnICAgICAgICAgICAgIm51bXB5IjogbnAuX192ZXJzaW9uX18sICJzY2lweSI6IHNjaXB5Ll9fdmVyc2lvbl9fLFxuJwogICAgICAgICcgICAgICAgICAgICAicGFuZGFzIjogcGQuX192ZXJzaW9uX18sICJjdWRhIjogdG9yY2gudmVyc2lvbi5jdWRhfScpCiAgICBzZXRfY2VsbChub3RlYm9vaywgMiwgc2V0dXApCgogICAgZmlsZXMgPSB7CiAgICAgICAgImxpdmVob29rL2Nvb3JkaW5hdG9yLnB5IjogRVhQRVJJTUVOVFMgLyAibGl2ZWhvb2siIC8gImNvb3JkaW5hdG9yLnB5IiwKICAgICAgICAidjExX2NhbmRpZGF0ZS9CZXRhVG91clNlYXJjaEhvb2suaCI6IEhFUkUgLyAiQmV0YVRvdXJTZWFyY2hIb29rLmgiLAogICAgICAgICJ2MTFfY2FuZGlkYXRlL0JldGFUb3VyVmlzaXQuaCI6IEhFUkUgLyAiQmV0YVRvdXJWaXNpdC5oIiwKICAgICAgICAidjExX2NhbmRpZGF0ZS9idWlsZF9zZWFyY2hfaG9vay5weSI6IEhFUkUgLyAiYnVpbGRfc2VhcmNoX2hvb2sucHkiLAogICAgICAgICJ2MTFfY2FuZGlkYXRlL3BvbGljeS5weSI6IEhFUkUgLyAicG9saWN5LnB5IiwKICAgICAgICAidjExX2NhbmRpZGF0ZS9waWxvdC5weSI6IEhFUkUgLyAicGlsb3QucHkiLAogICAgICAgICJ2MTFfY2FuZGlkYXRlL3J1bl9jZWxsLnB5IjogSEVSRSAvICJydW5fY2VsbC5weSIsCiAgICAgICAgInYxMV9jYW5kaWRhdGUvYW5hbHlzaXNfY2VsbC5weSI6IEhFUkUgLyAiYW5hbHlzaXNfY2VsbC5weSIsCiAgICAgICAgInYxMV9jYW5kaWRhdGUvYnVpbGRfY29sYWJfbm90ZWJvb2sucHkiOiBIRVJFIC8gImJ1aWxkX2NvbGFiX25vdGVib29rLnB5IiwKICAgIH0KICAgIGJ1bmRsZSA9IHtuYW1lOiBiYXNlNjQuYjY0ZW5jb2RlKHBhdGgucmVhZF9ieXRlcygpKS5kZWNvZGUoImFzY2lpIikKICAgICAgICAgICAgICBmb3IgbmFtZSwgcGF0aCBpbiBmaWxlcy5pdGVtcygpfQogICAgc2V0X2NlbGwobm90ZWJvb2ssIDMsICIiIkJVTkRMRURfRklMRVMgPSAiIiIgKyByZXByKGJ1bmRsZSkgKyAiIiIKQ09ERV9ST09UID0gV09SS19ST09UIC8gImNvZGUiCmZvciByZWwsIGVuY29kZWQgaW4gQlVORExFRF9GSUxFUy5pdGVtcygpOgogICAgY29udGVudCA9IGJhc2U2NC5iNjRkZWNvZGUoZW5jb2RlZCkKICAgIGZvciBkZXN0IGluIChDT0RFX1JPT1QgLyByZWwsIE9VVFBVVF9ST09UIC8gImNvZGUiIC8gcmVsKToKICAgICAgICBkZXN0LnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICAgICAgaWYgZGVzdC5leGlzdHMoKToKICAgICAgICAgICAgYXNzZXJ0IGRlc3QucmVhZF9ieXRlcygpID09IGNvbnRlbnQsIGYiRGVuZXkga29kdSBkZcSfacWfbWnFnzoge3JlbH0iCiAgICAgICAgZWxzZToKICAgICAgICAgICAgZGVzdC53cml0ZV9ieXRlcyhjb250ZW50KQpDT0RFX1NIQTI1NiA9IHtyZWw6IGhhc2hsaWIuc2hhMjU2KGJhc2U2NC5iNjRkZWNvZGUoZW5jb2RlZCkpLmhleGRpZ2VzdCgpCiAgICAgICAgICAgICAgIGZvciByZWwsIGVuY29kZWQgaW4gQlVORExFRF9GSUxFUy5pdGVtcygpfQpzeXMucGF0aFs6MF0gPSBbc3RyKENPREVfUk9PVCAvICJsaXZlaG9vayIpLCBzdHIoQ09ERV9ST09UIC8gInYxMV9jYW5kaWRhdGUiKV0KaW1wb3J0IGJ1aWxkX3NlYXJjaF9ob29rCmZyb20gY29vcmRpbmF0b3IgaW1wb3J0IHJlYWRfdG91cgpmcm9tIHBpbG90IGltcG9ydCBydW5fZ3VpZGVkX2FybSwgcnVuX25hdGl2ZV9hcm0sIHNoYTI1Nl9maWxlCmZyb20gcG9saWN5IGltcG9ydCBQT0xJQ1lfVkVSU0lPTgpwcmludCgiRG9uZHVydWxtdcWfIGRlbmV5IGtvZHU6IiwgQ09ERV9TSEEyNTYpCiIiIikKCiAgICBidWlsZCA9IG9sZCg0KQogICAgYnVpbGQgPSByZXBsYWNlX29uY2UoYnVpbGQsICdQQVRDSEVEX0RJUiA9IExLSF9ST09UIC8gIkxLSC0zLjAuMTQtQmV0YVRvdXIiJywKICAgICAgICAgICAgICAgICAgICAgICAgICdQQVRDSEVEX0RJUiA9IExLSF9ST09UIC8gIkxLSC0zLjAuMTQtQmV0YVRvdXItdjExcjIiJykKICAgIGJ1aWxkID0gcmVwbGFjZV9vbmNlKGJ1aWxkLCAnYnVpbGRfbGl2ZWhvb2suYnVpbGQoU09VUkNFX0RJUiwgUEFUQ0hFRF9ESVIsIGpvYnM9bWluKDYsIENQVV9DT1VOVCkpJywKICAgICAgICAgICAgICAgICAgICAgICAgICdidWlsZF9zZWFyY2hfaG9vay5idWlsZChTT1VSQ0VfRElSLCBQQVRDSEVEX0RJUiwgam9icz1taW4oNiwgQ1BVX0NPVU5UKSknKQogICAgYnVpbGQgPSByZXBsYWNlX29uY2UoYnVpbGQsCiAgICAgICAgJ2lmIG5vdCBTT1VSQ0VfSU5GT19QQVRILmV4aXN0cygpOicsCiAgICAgICAgJ2ZvciBmaWxlbmFtZSwgZXhwZWN0ZWQgaW4gYnVpbGRfc2VhcmNoX2hvb2suVVBTVFJFQU1fU0hBMjU2Lml0ZW1zKCk6XG4nCiAgICAgICAgJyAgICBhc3NlcnQgaGFzaGxpYi5zaGEyNTYoKFNPVVJDRV9ESVIgLyAiU1JDIiAvIGZpbGVuYW1lKS5yZWFkX2J5dGVzKCkpLmhleGRpZ2VzdCgpID09IGV4cGVjdGVkLCBmaWxlbmFtZVxuJwogICAgICAgICdpZiBub3QgU09VUkNFX0lORk9fUEFUSC5leGlzdHMoKTonKQogICAgYnVpbGQgPSByZXBsYWNlX29uY2UoYnVpbGQsCiAgICAgICAgJ0JVSUxEX0lORk8gPSB7Im5hdGl2ZV9zaGEyNTYiOiBzaGEyNTZfZmlsZShOQVRJVkVfTEtIKSwnLAogICAgICAgICdwYXRjaF9tYW5pZmVzdCA9IGpzb24ubG9hZHMoKFBBVENIRURfRElSIC8gImJldGF0b3VyX2J1aWxkLmpzb24iKS5yZWFkX3RleHQoKSlcbicKICAgICAgICAnYXNzZXJ0IHBhdGNoX21hbmlmZXN0WyJ1cHN0cmVhbV9zaGEyNTYiXSA9PSBidWlsZF9zZWFyY2hfaG9vay5VUFNUUkVBTV9TSEEyNTZcbicKICAgICAgICAnYXNzZXJ0IHBhdGNoX21hbmlmZXN0WyJzZWFyY2hfaG9va19zaGEyNTYiXSA9PSBDT0RFX1NIQTI1NlsidjExX2NhbmRpZGF0ZS9CZXRhVG91clNlYXJjaEhvb2suaCJdXG4nCiAgICAgICAgJ2Fzc2VydCBwYXRjaF9tYW5pZmVzdFsidmlzaXRfaGVhZGVyX3NoYTI1NiJdID09IENPREVfU0hBMjU2WyJ2MTFfY2FuZGlkYXRlL0JldGFUb3VyVmlzaXQuaCJdXG4nCiAgICAgICAgJ2Fzc2VydCBwYXRjaF9tYW5pZmVzdFsiYmluYXJ5X3NoYTI1NiJdID09IHNoYTI1Nl9maWxlKFBBVENIRURfTEtIKVxuJwogICAgICAgICdCVUlMRF9JTkZPID0geyJuYXRpdmVfc2hhMjU2Ijogc2hhMjU2X2ZpbGUoTkFUSVZFX0xLSCksJykKICAgIGJ1aWxkID0gcmVwbGFjZV9vbmNlKGJ1aWxkLAogICAgICAgICcidXBzdHJlYW1fZmluZHRvdXJfc2hhMjU2IjogTEtIX0ZJTkRUT1VSX1NIQTI1Nn0nLAogICAgICAgICcidXBzdHJlYW1fZmluZHRvdXJfc2hhMjU2IjogTEtIX0ZJTkRUT1VSX1NIQTI1NixcbicKICAgICAgICAnICAgICAgICAgICAgICAiYWN0aW9uX3Nsb3RfcnVsZSI6IHBhdGNoX21hbmlmZXN0WyJhY3Rpb25fc2xvdF9ydWxlIl0sXG4nCiAgICAgICAgJyAgICAgICAgICAgICAgImNhbmRpZGF0ZV9yZW9yZGVyaW5nIjogcGF0Y2hfbWFuaWZlc3RbInVwc3RyZWFtX2NhbmRpZGF0ZV9yZW9yZGVyaW5nIl19JykKICAgIGJ1aWxkID0gcmVwbGFjZV9vbmNlKGJ1aWxkLCAncHJpbnQoIkxLSCAzLjAuMTQgZGVybGVuZGk6IiwgQlVJTERfSU5GTyknLAogICAgICAgICdwcmludCgiTEtIIDMuMC4xNCBkZXJsZW5kaToiLCB7azogdiBmb3IgaywgdiBpbiBCVUlMRF9JTkZPLml0ZW1zKCkgaWYgayAhPSAiY2FuZGlkYXRlX3Jlb3JkZXJpbmcifSlcbicKICAgICAgICAncHJpbnQoIkxLSCBhZGF5IHPEsXJhc8SxIHllbmlkZW4gZMO8emVubGVtZSBrdXJhbMSxIChrYXluYWsgZGVuZXRpbWkpOiIsXG4nCiAgICAgICAgJyAgICAgIGpzb24uZHVtcHMoQlVJTERfSU5GT1siY2FuZGlkYXRlX3Jlb3JkZXJpbmciXSwgaW5kZW50PTEpKScpCiAgICBzZXRfY2VsbChub3RlYm9vaywgNCwgYnVpbGQpCgogICAgc2V0X2NlbGwobm90ZWJvb2ssIDUsICIiImRlZiB3cml0ZV9qc29uX2F0b21pYyhwYXRoLCBvYmopOgogICAgcGF0aCA9IFBhdGgocGF0aCkKICAgIHBhdGgucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHRtcCA9IHBhdGgud2l0aF9uYW1lKHBhdGgubmFtZSArICIudG1wIikKICAgIHRtcC53cml0ZV90ZXh0KGpzb24uZHVtcHMob2JqLCBpbmRlbnQ9MiwgZW5zdXJlX2FzY2lpPUZhbHNlLCBhbGxvd19uYW49RmFsc2UpKQogICAgb3MucmVwbGFjZSh0bXAsIHBhdGgpCgpNQU5JRkVTVCA9IHsic2NoZW1hIjogMSwgInByb2ZpbGUiOiAiZm91cl9yZXVzZWRfMTBrX2RldmVsb3BtZW50X2luc3RhbmNlcyIsCiAgICAgICAgICAgICJleHBlcmltZW50X25hbWUiOiBFWFBFUklNRU5UX05BTUUsICJhcm1zIjogQVJNUywgInNlZWRzIjogU0VFRFMsCiAgICAgICAgICAgICJpbnN0YW5jZV9zcGVjcyI6IElOU1RBTkNFX1NQRUNTLCAieHlfaGFzaGVzIjogRVhQRUNURURfWFlfU0hBMjU2LAogICAgICAgICAgICAidGltZV90b190YXJnZXRfY29zdHNfdjEwX25hdGl2ZV9wb3A2X2Jlc3Rfb2Zfc2l4IjogVEFSR0VUX0NPU1RTLAogICAgICAgICAgICAicG9saWN5X3ZlcnNpb24iOiBQT0xJQ1lfVkVSU0lPTiwKICAgICAgICAgICAgInNwYXJzZV9mcmFjdGlvbiI6IFNQQVJTRV9GUkFDVElPTiwgImdyYXBoX2siOiBHUkFQSF9LLAogICAgICAgICAgICAicmVwb19jb21taXQiOiBSRVBPX0NPTU1JVCwgImNoZWNrcG9pbnRfc2hhMjU2IjogQ0hFQ0tQT0lOVF9TSEEyNTYsCiAgICAgICAgICAgICJoYXJkd2FyZSI6IEhBUkRXQVJFLCAiY29kZV9zaGEyNTYiOiBDT0RFX1NIQTI1NiwgImJ1aWxkIjogQlVJTERfSU5GTywKICAgICAgICAgICAgImFuYWx5c2lzX3BsYW4iOiAiTWVjaGFuaXNtIHBpbG90IG9uIHJldXNlZCBjYXNlczsgbm8gcGFwZXIgcGVyZm9ybWFuY2UgY2xhaW0gYmVmb3JlIGZyb3plbiBpbmRlcGVuZGVudCBob2xkb3V0LiJ9Cm1hbmlmZXN0X3BhdGggPSBPVVRQVVRfUk9PVCAvICJtYW5pZmVzdC5qc29uIgppZiBtYW5pZmVzdF9wYXRoLmV4aXN0cygpOgogICAgc2F2ZWQgPSBqc29uLmxvYWRzKG1hbmlmZXN0X3BhdGgucmVhZF90ZXh0KCkpCiAgICBmb3Iga2V5IGluIE1BTklGRVNUOgogICAgICAgIGlmIGtleSAhPSAiaGFyZHdhcmUiOgogICAgICAgICAgICBhc3NlcnQgc2F2ZWRba2V5XSA9PSBNQU5JRkVTVFtrZXldLCBmIkRyaXZlJ2Rha2kgcGlsb3R1biB7a2V5fSBheWFyxLEgZGXEn2nFn3RpOyB5ZW5pIGRlbmV5IGFkxLEga3VsbGFuxLFuLiIKICAgIGZvciBrZXkgaW4gKCJncHUiLCAidmNwdXMiKToKICAgICAgICBhc3NlcnQgc2F2ZWRbImhhcmR3YXJlIl1ba2V5XSA9PSBIQVJEV0FSRVtrZXldLCBmIkRvbmFuxLFtIHtrZXl9IGRlxJ9pxZ90aTsgZcWfbGkgcGlsb3R1IGF5bsSxIHTDvHJkZSDDp2FsxLHFn3TEsXLEsW4uIgplbHNlOgogICAgd3JpdGVfanNvbl9hdG9taWMobWFuaWZlc3RfcGF0aCwgTUFOSUZFU1QpCgojIEdQVSB3YXJtdXAgYW5kIGV4YWN0IGZpcnN0LXRyaWFsL25hdGl2ZSBlcXVpdmFsZW5jZSBjaGVjay4Kd2FybV94eSA9IFQudG9faW50X2Nvb3JkcyhULmdlbl9pbnN0YW5jZSgxMDAsICJ1bmlmb3JtIiwgbnAucmFuZG9tLmRlZmF1bHRfcm5nKDkwOTApKSkKd2FybV9zdWNjID0gbnAuc3RhY2soW25wLnJvbGwobnAuYXJhbmdlKDEwMCksIC0xKV0gKiA2LCBheGlzPTEpLmFzdHlwZShucC5pbnQzMikKTS5wcmVkaWN0X3RvdXJmb2xkKE1PREVMLCB3YXJtX3h5LCBHUkFQSF9LLCBERVZJQ0UsCiAgICAgICAgICAgICAgICAgICBlbnNfc3VjYz13YXJtX3N1Y2MsIGVuc19jb3N0PW5wLm9uZXMoNikpCndhcm1fcHJvYmxlbSA9IFdPUktfUk9PVCAvICJ3YXJtdXAudHNwIgpULndyaXRlX3RzcCh3YXJtX3Byb2JsZW0sIHdhcm1feHkpCndhcm1fZGlyID0gV09SS19ST09UIC8gIndhcm11cCIgLyB1dWlkLnV1aWQ0KCkuaGV4Cndhcm1fZGlyLnBhcmVudC5ta2RpcihleGlzdF9vaz1UcnVlKQp3YXJtX2NhY2hlID0gV09SS19ST09UIC8gIndhcm11cCIgLyAoImNhY2hlXyIgKyB1dWlkLnV1aWQ0KCkuaGV4KQp3YXJtID0gcnVuX2d1aWRlZF9hcm0oInNoYW0iLCBQQVRDSEVEX0xLSCwgd2FybV9wcm9ibGVtLCB3YXJtX3h5LCB3YXJtX2RpciwKICAgICAgICAgICAgICAgICAgICAgIFNFRURTLCAxNS4wLCBNT0RFTCwgREVWSUNFLCBULCBNLAogICAgICAgICAgICAgICAgICAgICAgZnJhY3Rpb249U1BBUlNFX0ZSQUNUSU9OLCBncmFwaF9rPUdSQVBIX0ssCiAgICAgICAgICAgICAgICAgICAgICBkZWNpc2lvbl9jYWNoZV9kaXI9d2FybV9jYWNoZSwgbW9kZWxfaWRlbnRpdHk9Q0hFQ0tQT0lOVF9TSEEyNTYpCndhcm1fZ3VpZGVkX2RpciA9IFdPUktfUk9PVCAvICJ3YXJtdXAiIC8gdXVpZC51dWlkNCgpLmhleAp3YXJtX2d1aWRlZCA9IHJ1bl9ndWlkZWRfYXJtKCJiZXRhX2FkZCIsIFBBVENIRURfTEtILCB3YXJtX3Byb2JsZW0sIHdhcm1feHksIHdhcm1fZ3VpZGVkX2RpciwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICBTRUVEUywgMTAuMCwgTU9ERUwsIERFVklDRSwgVCwgTSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICBmcmFjdGlvbj1TUEFSU0VfRlJBQ1RJT04sIGdyYXBoX2s9R1JBUEhfSywKICAgICAgICAgICAgICAgICAgICAgICAgICAgICBkZWNpc2lvbl9jYWNoZV9kaXI9d2FybV9jYWNoZSwgbW9kZWxfaWRlbnRpdHk9Q0hFQ0tQT0lOVF9TSEEyNTYpCmFzc2VydCB3YXJtWyJwb2xpY3lfYXVkaXQiXVsiZGVjaXNpb25fYmFzaXMiXSA9PSAiZnJlc2hfaW5mZXJlbmNlIgphc3NlcnQgd2FybV9ndWlkZWRbInBvbGljeV9hdWRpdCJdWyJkZWNpc2lvbl9iYXNpcyJdID09ICJjYWNoZWRfaW5mZXJlbmNlIiwgIkF5csSxIGtvbGxhcsSxbiBtb2RlbCBnaXJkaXNpIGXFn2xlxZ9tZWRpIgphc3NlcnQgd2FybV9ndWlkZWRbImZpcnN0X3RvdXJfaGFzaGVzIl0gPT0gd2FybVsiZmlyc3RfdG91cl9oYXNoZXMiXSwgIkF5csSxIGtvbGxhcsSxbiBpbGsgdHVydSBlxZ9sZcWfbWVkaSIKYXNzZXJ0IHN1bSh3YXJtX2d1aWRlZFsicG9saWN5X2F1ZGl0Il1bInNlbGVjdGVkX2FjdGlvbnNfcGVyX3NlZWQiXSkgPiAwLCAiSXPEsW5tYSBrb2x1bmRhIGV5bGVtIHNlw6dpbG1lZGkiCmZvciBzZWVkIGluIFNFRURTOgogICAgaG9va19rZXlzID0gKHdhcm1fZ3VpZGVkX2RpciAvIGYic2VlZF97c2VlZH0iIC8gInRlbGVtZXRyeS50eHQiKS5yZWFkX3RleHQoKQogICAgYXNzZXJ0ICJzbG90X2Zsb29yX2VuZm9yY2VkIDEiIGluIGhvb2tfa2V5cyBhbmQgInNsb3RfcmVzdG9yYXRpb25zIiBpbiBob29rX2tleXMKZGVmIGVkZ2Vfc2V0KHRvdXIpOgogICAgdG91ciA9IG5wLmFzYXJyYXkodG91ciwgZHR5cGU9bnAuaW50MzIpCiAgICByZXR1cm4gc2V0KHppcChucC5taW5pbXVtKHRvdXIsIG5wLnJvbGwodG91ciwgLTEpKSwKICAgICAgICAgICAgICAgICAgIG5wLm1heGltdW0odG91ciwgbnAucm9sbCh0b3VyLCAtMSkpKSkKZm9yIHNlZWQgaW4gU0VFRFM6CiAgICBuYXRpdmUgPSBULnJ1bl9sa2goc3RyKE5BVElWRV9MS0gpLCBzdHIod2FybV9wcm9ibGVtKSwKICAgICAgICAgICAgICAgICAgICAgICB7IkNBTkRJREFURV9TRVRfVFlQRSI6ICJQT1BNVVNJQyIsICJNQVhfQ0FORElEQVRFUyI6IDYsCiAgICAgICAgICAgICAgICAgICAgICAgICJTVUJHUkFESUVOVCI6ICJOTyIsICJTRUVEIjogc2VlZCwgIk1BWF9UUklBTFMiOiAxLAogICAgICAgICAgICAgICAgICAgICAgICAiVElNRV9MSU1JVCI6IDMwLCAiU1RPUF9BVF9PUFRJTVVNIjogIk5PIn0sCiAgICAgICAgICAgICAgICAgICAgICAgc3RyKFdPUktfUk9PVCAvICJ3YXJtdXAiIC8gZiJuYXRpdmVfe3NlZWR9X3t1dWlkLnV1aWQ0KCkuaGV4fSIpLCB0aW1lb3V0PTMwKQogICAgZmlyc3QgPSBucC5hc2FycmF5KHJlYWRfdG91cih3YXJtX2RpciAvIGYic2VlZF97c2VlZH0iIC8gImZpcnN0LnRvdXIiKSkgLSAxCiAgICBhc3NlcnQgZWRnZV9zZXQobmF0aXZlWyJ0b3VyIl0pID09IGVkZ2Vfc2V0KGZpcnN0KSwgZiLEsGxrIHR1ciB1eXXFn211eW9yOiB7c2VlZH0iCnByaW50KCJHUFUsIFBPUDYgeWFtYXPEsSB2ZSBpbGsgdHVyIGXFn2l0bGnEn2kgZG/En3J1bGFuZMSxLiIpCiIiIikKCiAgICBpbnN0YW5jZXMgPSBvbGQoNikKICAgIGluc3RhbmNlcyA9IHJlcGxhY2Vfb25jZShpbnN0YW5jZXMsICdhc3NlcnQgbGVuKElOU1RBTkNFUykgPT0gOCcsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgJ2Fzc2VydCBsZW4oSU5TVEFOQ0VTKSA9PSA0JykKICAgIHNldF9jZWxsKG5vdGVib29rLCA2LCBpbnN0YW5jZXMpCiAgICBzZXRfY2VsbChub3RlYm9vaywgNywgKEhFUkUgLyAicnVuX2NlbGwucHkiKS5yZWFkX3RleHQoKSkKICAgIHNldF9jZWxsKG5vdGVib29rLCA4LCAoSEVSRSAvICJhbmFseXNpc19jZWxsLnB5IikucmVhZF90ZXh0KCkpCgogICAgYXJjaGl2ZSA9IG9sZCg5KQogICAgYXJjaGl2ZSA9IHJlcGxhY2Vfb25jZShhcmNoaXZlLAogICAgICAgICd7Inh5Lm5weiIsICJuYXRpdmUuY2FuZGlkYXRlcyIsICJsaXZlLmNhbmRpZGF0ZXMiLCAicG9zdC5jYW5kaWRhdGVzIiwgImZpcnN0LnRvdXIiLCAiYmVzdC50b3VyIiwgInN0ZG91dC50eHQiLCAic3RkZXJyLnR4dCIsICJwb2xpY3lfYXJyYXlzLm5weiJ9JywKICAgICAgICAneyJ4eS5ucHoiLCAibmF0aXZlLmNhbmRpZGF0ZXMiLCAibGl2ZS5jYW5kaWRhdGVzIiwgInBvc3QuY2FuZGlkYXRlcyIsICJmaXJzdC50b3VyIiwgImJlc3QudG91ciIsICJhY3Rpb24udHh0IiwgInN0ZG91dC50eHQiLCAic3RkZXJyLnR4dCIsICJwb2xpY3lfYXJyYXlzLm5weiJ9JykKICAgIGFyY2hpdmUgPSByZXBsYWNlX29uY2UoYXJjaGl2ZSwgJyJwb2xpY3lfYXVkaXQiIGluIHJlbC5wYXJ0cycsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICcicG9saWN5X2F1ZGl0IiBpbiByZWwucGFydHMgb3IgImRlY2lzaW9uX2NhY2hlIiBpbiByZWwucGFydHMnKQogICAgYXJjaGl2ZSA9IHJlcGxhY2Vfb25jZShhcmNoaXZlLCAnY29tcGxldGUgPSBsZW4ocm93cykgPT0gOCcsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICdjb21wbGV0ZSA9IGxlbihyb3dzKSA9PSBsZW4oSU5TVEFOQ0VTKScpCiAgICBhcmNoaXZlID0gcmVwbGFjZV9vbmNlKGFyY2hpdmUsCiAgICAgICAgJ3RhbWFtbGFuYW4gw7ZybmVrIHtsZW4ocm93cyl9LzguJywKICAgICAgICAndGFtYW1sYW5hbiDDtnJuZWsge2xlbihyb3dzKX0ve2xlbihJTlNUQU5DRVMpfS4nKQogICAgc2V0X2NlbGwobm90ZWJvb2ssIDksIGFyY2hpdmUpCgogICAgbm90ZWJvb2tbIm1ldGFkYXRhIl1bImNvbGFiIl1bIm5hbWUiXSA9IE9VVFBVVC5uYW1lCiAgICByZXR1cm4gbm90ZWJvb2sKCgpkZWYgbWFpbigpIC0+IE5vbmU6CiAgICBub3RlYm9vayA9IG1ha2Vfbm90ZWJvb2soKQogICAgT1VUUFVULnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBPVVRQVVQud3JpdGVfdGV4dChqc29uLmR1bXBzKG5vdGVib29rLCBlbnN1cmVfYXNjaWk9RmFsc2UsIGluZGVudD0xKSArICJcbiIpCiAgICBwcmludChPVVRQVVQpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo='}
CODE_ROOT = WORK_ROOT / "code"
for rel, encoded in BUNDLED_FILES.items():
    content = base64.b64decode(encoded)
    for dest in (CODE_ROOT / rel, OUTPUT_ROOT / "code" / rel):
        dest.parent.mkdir(parents=True, exist_ok=True)
        if dest.exists():
            assert dest.read_bytes() == content, f"Deney kodu değişmiş: {rel}"
        else:
            dest.write_bytes(content)
CODE_SHA256 = {rel: hashlib.sha256(base64.b64decode(encoded)).hexdigest()
               for rel, encoded in BUNDLED_FILES.items()}
sys.path[:0] = [str(CODE_ROOT / "livehook"), str(CODE_ROOT / "v11_candidate")]
import build_search_hook
from coordinator import read_tour
from pilot import run_guided_arm, run_native_arm, sha256_file
from policy import POLICY_VERSION
print("Dondurulmuş deney kodu:", CODE_SHA256)


Dondurulmuş deney kodu: {'livehook/coordinator.py': 'd6064841d1ab6d7a0ae00ddd33d142934349df11c75ce76741466df9c9d43a56', 'v11_candidate/BetaTourSearchHook.h': '386928f2f80ba35efe925b40fa3012062872f88dffe67acf6f125befcc586224', 'v11_candidate/BetaTourVisit.h': '6c133c2875a2b71a81de93a3ee4e00301898fe0db7022a464577e15de5210b51', 'v11_candidate/build_search_hook.py': 'a9f5f777278e0cf8e70142454974d6e5e7c3166d5a46df8a573e9b53ce983651', 'v11_candidate/policy.py': 'c450cc0cdcb924cd504e992aab05a890f90fce77a8e6e0af6bb56676cd619513', 'v11_candidate/pilot.py': '8bb4272bfe045721bbecc2ab667402b9e6e427327a12b82927d317ab757b48af', 'v11_candidate/run_cell.py': '6069a94571dd9dd59ef6ae6b87d119b3c4ff85b0699e725909f7b3a664724ed4', 'v11_candidate/analysis_cell.py': 'a011dc9b964a51699ec37e792596a8a14a66f61343b29dd234c7521aeb4a33c5', 'v11_candidate/build_colab_notebook.py': 'e54afd896ea10d0e17ea9ffccfd08edf1b154e284600b1d4eb20bbd7e5682a5c'}


In [ ]:
LKH_ROOT = WORK_ROOT / "lkh"
LKH_ROOT.mkdir(exist_ok=True)
SOURCE_DIR = LKH_ROOT / "LKH-3.0.14"
SOURCE_INFO_PATH = LKH_ROOT / "source_info.json"
if not SOURCE_DIR.exists():
    sources = [
        ("official", "https://webhotel4.ruc.dk/~keld/research/LKH-3/LKH-3.0.14.tgz", None, 12),
        ("pinned_mirror", "https://raw.githubusercontent.com/ZhekaiChen/ALCO/main/third_party/LKH3/LKH-3.0.14.tar", LKH_TAR_SHA256, 90),
    ]
    last_error = None
    for label, url, expected_sha, timeout_s in sources:
        archive = LKH_ROOT / ("LKH-3.0.14.tgz" if url.endswith(".tgz") else "LKH-3.0.14.tar")
        try:
            request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(request, timeout=timeout_s) as response, archive.open("wb") as out:
                shutil.copyfileobj(response, out)
            digest = hashlib.sha256(archive.read_bytes()).hexdigest()
            if expected_sha:
                assert digest == expected_sha, f"LKH arşiv SHA256 uyuşmuyor: {digest}"
            with tarfile.open(archive, "r:*") as tar:
                try:
                    tar.extractall(LKH_ROOT, filter="data")
                except TypeError:
                    tar.extractall(LKH_ROOT)
            assert SOURCE_DIR.exists()
            actual_source_sha = hashlib.sha256((SOURCE_DIR / "SRC/FindTour.c").read_bytes()).hexdigest()
            assert actual_source_sha == LKH_FINDTOUR_SHA256, f"LKH 3.0.14 kaynağı beklenenden farklı: {actual_source_sha}"
            SOURCE_INFO_PATH.write_text(json.dumps({"label": label, "url": url, "archive_sha256": digest}, indent=2))
            break
        except Exception as exc:
            last_error = exc
            if SOURCE_DIR.exists():
                shutil.rmtree(SOURCE_DIR)
    else:
        raise RuntimeError(f"LKH indirilemedi: {last_error}")
assert hashlib.sha256((SOURCE_DIR / "SRC/FindTour.c").read_bytes()).hexdigest() == LKH_FINDTOUR_SHA256
for filename, expected in build_search_hook.UPSTREAM_SHA256.items():
    assert hashlib.sha256((SOURCE_DIR / "SRC" / filename).read_bytes()).hexdigest() == expected, filename
if not SOURCE_INFO_PATH.exists():
    cached = next((p for p in (LKH_ROOT / "LKH-3.0.14.tar", LKH_ROOT / "LKH-3.0.14.tgz") if p.exists()), None)
    assert cached is not None, "LKH kaynak kayıt dosyası eksik; kaynak arşiv bulunamadı"
    SOURCE_INFO_PATH.write_text(json.dumps({"label": "recovered_cache", "url": "local_cache",
                                            "archive_sha256": hashlib.sha256(cached.read_bytes()).hexdigest()}, indent=2))
NATIVE_LKH = SOURCE_DIR / "LKH"
if not NATIVE_LKH.exists():
    with (LKH_ROOT / "native_build.log").open("w") as build_log:
        subprocess.run(["make", "-j", str(min(6, CPU_COUNT))], cwd=SOURCE_DIR, check=True,
                       stdout=build_log, stderr=subprocess.STDOUT)
PATCHED_DIR = LKH_ROOT / "LKH-3.0.14-BetaTour-v11r2"
PATCHED_LKH = PATCHED_DIR / "LKH"
if not PATCHED_LKH.exists():
    if PATCHED_DIR.exists():
        shutil.rmtree(PATCHED_DIR)
    build_search_hook.build(SOURCE_DIR, PATCHED_DIR, jobs=min(6, CPU_COUNT))
patch_manifest = json.loads((PATCHED_DIR / "betatour_build.json").read_text())
assert patch_manifest["upstream_sha256"] == build_search_hook.UPSTREAM_SHA256
assert patch_manifest["search_hook_sha256"] == CODE_SHA256["v11_candidate/BetaTourSearchHook.h"]
assert patch_manifest["visit_header_sha256"] == CODE_SHA256["v11_candidate/BetaTourVisit.h"]
assert patch_manifest["binary_sha256"] == sha256_file(PATCHED_LKH)
BUILD_INFO = {"native_sha256": sha256_file(NATIVE_LKH),
              "patched_sha256": sha256_file(PATCHED_LKH),
              "upstream_findtour_sha256": LKH_FINDTOUR_SHA256,
              "action_slot_rule": patch_manifest["action_slot_rule"],
              "candidate_reordering": patch_manifest["upstream_candidate_reordering"]}
origin_path = OUTPUT_ROOT / "code" / "lkh_source_info.json"
if not origin_path.exists():
    origin_path.write_text(SOURCE_INFO_PATH.read_text())
print("LKH 3.0.14 derlendi:", {k: v for k, v in BUILD_INFO.items() if k != "candidate_reordering"})
print("LKH aday sırası yeniden düzenleme kuralı (kaynak denetimi):",
      json.dumps(BUILD_INFO["candidate_reordering"], indent=1))


LKH 3.0.14 derlendi: {'native_sha256': '3d0109989a944bd25e15512aff26097b5761ceb6a98ab7bc87ba520cc168d0cd', 'patched_sha256': '3bd8587f0bedffe5ab6e08c9edcb3e64b928711f9bccd84f041429a3ba7c45f2', 'upstream_findtour_sha256': '567e72acc9b7392488dd158be4df0267560424204a0e88d2d9b62f923a7c0eee', 'action_slot_rule': 'floor enforced at every completed-trial boundary'}
LKH aday sırası yeniden düzenleme kuralı (kaynak denetimi): {
 "FindTour_sha256": "567e72acc9b7392488dd158be4df0267560424204a0e88d2d9b62f923a7c0eee",
 "AdjustCandidateSet_sha256": "930e08d9ea5add44b276c2464daa858a9e59b91857df50ce4d211375de93fe86",
 "ResetCandidateSet_sha256": "41b546f0ea5034f8023f7adb4112d9c1be8fab977991fbcb7926e6bba4a9f6a4",
 "findtour_trial_loop_calls_adjust_candidate_set": true,
 "adjust_moves_edges_of_two_best_tours_to_front": true,
 "adjust_sorts_rows_by_alpha_then_cost": false,
 "reset_sorts_rows_by_alpha_then_cost": true
}


In [ ]:
def write_json_atomic(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, ensure_ascii=False, allow_nan=False))
    os.replace(tmp, path)

MANIFEST = {"schema": 1, "profile": "four_reused_10k_development_instances",
            "experiment_name": EXPERIMENT_NAME, "arms": ARMS, "seeds": SEEDS,
            "instance_specs": INSTANCE_SPECS, "xy_hashes": EXPECTED_XY_SHA256,
            "time_to_target_costs_v10_native_pop6_best_of_six": TARGET_COSTS,
            "policy_version": POLICY_VERSION,
            "sparse_fraction": SPARSE_FRACTION, "graph_k": GRAPH_K,
            "repo_commit": REPO_COMMIT, "checkpoint_sha256": CHECKPOINT_SHA256,
            "hardware": HARDWARE, "code_sha256": CODE_SHA256, "build": BUILD_INFO,
            "analysis_plan": "Mechanism pilot on reused cases; no paper performance claim before frozen independent holdout."}
manifest_path = OUTPUT_ROOT / "manifest.json"
if manifest_path.exists():
    saved = json.loads(manifest_path.read_text())
    for key in MANIFEST:
        if key != "hardware":
            assert saved[key] == MANIFEST[key], f"Drive'daki pilotun {key} ayarı değişti; yeni deney adı kullanın."
    for key in ("gpu", "vcpus"):
        assert saved["hardware"][key] == HARDWARE[key], f"Donanım {key} değişti; eşli pilotu aynı türde çalıştırın."
else:
    write_json_atomic(manifest_path, MANIFEST)

# GPU warmup and exact first-trial/native equivalence check.
warm_xy = T.to_int_coords(T.gen_instance(100, "uniform", np.random.default_rng(9090)))
warm_succ = np.stack([np.roll(np.arange(100), -1)] * 6, axis=1).astype(np.int32)
M.predict_tourfold(MODEL, warm_xy, GRAPH_K, DEVICE,
                   ens_succ=warm_succ, ens_cost=np.ones(6))
warm_problem = WORK_ROOT / "warmup.tsp"
T.write_tsp(warm_problem, warm_xy)
warm_dir = WORK_ROOT / "warmup" / uuid.uuid4().hex
warm_dir.parent.mkdir(exist_ok=True)
warm_cache = WORK_ROOT / "warmup" / ("cache_" + uuid.uuid4().hex)
warm = run_guided_arm("sham", PATCHED_LKH, warm_problem, warm_xy, warm_dir,
                      SEEDS, 15.0, MODEL, DEVICE, T, M,
                      fraction=SPARSE_FRACTION, graph_k=GRAPH_K,
                      decision_cache_dir=warm_cache, model_identity=CHECKPOINT_SHA256)
warm_guided_dir = WORK_ROOT / "warmup" / uuid.uuid4().hex
warm_guided = run_guided_arm("beta_add", PATCHED_LKH, warm_problem, warm_xy, warm_guided_dir,
                             SEEDS, 10.0, MODEL, DEVICE, T, M,
                             fraction=SPARSE_FRACTION, graph_k=GRAPH_K,
                             decision_cache_dir=warm_cache, model_identity=CHECKPOINT_SHA256)
assert warm["policy_audit"]["decision_basis"] == "fresh_inference"
assert warm_guided["policy_audit"]["decision_basis"] == "cached_inference", "Ayrı kolların model girdisi eşleşmedi"
assert warm_guided["first_tour_hashes"] == warm["first_tour_hashes"], "Ayrı kolların ilk turu eşleşmedi"
assert sum(warm_guided["policy_audit"]["selected_actions_per_seed"]) > 0, "Isınma kolunda eylem seçilmedi"
for seed in SEEDS:
    hook_keys = (warm_guided_dir / f"seed_{seed}" / "telemetry.txt").read_text()
    assert "slot_floor_enforced 1" in hook_keys and "slot_restorations" in hook_keys
def edge_set(tour):
    tour = np.asarray(tour, dtype=np.int32)
    return set(zip(np.minimum(tour, np.roll(tour, -1)),
                   np.maximum(tour, np.roll(tour, -1))))
for seed in SEEDS:
    native = T.run_lkh(str(NATIVE_LKH), str(warm_problem),
                       {"CANDIDATE_SET_TYPE": "POPMUSIC", "MAX_CANDIDATES": 6,
                        "SUBGRADIENT": "NO", "SEED": seed, "MAX_TRIALS": 1,
                        "TIME_LIMIT": 30, "STOP_AT_OPTIMUM": "NO"},
                       str(WORK_ROOT / "warmup" / f"native_{seed}_{uuid.uuid4().hex}"), timeout=30)
    first = np.asarray(read_tour(warm_dir / f"seed_{seed}" / "first.tour")) - 1
    assert edge_set(native["tour"]) == edge_set(first), f"İlk tur uyuşmuyor: {seed}"
print("GPU, POP6 yaması ve ilk tur eşitliği doğrulandı.")


GPU, POP6 yaması ve ilk tur eşitliği doğrulandı.


In [ ]:
INSTANCES = []
for group, n, count, distribution, generator_seed, budget_s in INSTANCE_SPECS:
    rng = np.random.default_rng(generator_seed)
    for index in range(count):
        xy = T.to_int_coords(T.gen_instance(n, distribution, rng))
        name = f"augment_{group}_{index:03d}"
        digest = hashlib.sha256(np.ascontiguousarray(xy).tobytes()).hexdigest()
        assert digest == EXPECTED_XY_SHA256[name], f"Eski v8 koordinatları uyuşmuyor: {name}"
        INSTANCES.append({"name": name, "group": group, "n": n,
                          "budget_s": budget_s, "xy": xy, "xy_sha256": digest})
INSTANCES.sort(key=lambda item: (item["budget_s"], item["name"]))
assert len(INSTANCES) == 4
def arm_order(name):
    return sorted(ARMS, key=lambda arm: hashlib.sha256((name + ":" + arm).encode()).hexdigest())
print("Pilot:", len(INSTANCES), "örnek,", len(ARMS), "kol; kesintisiz tek kol bütçesi",
      sum(item["budget_s"] for item in INSTANCES) / 3600, "saat")


Pilot: 4 örnek, 6 kol; kesintisiz tek kol bütçesi 0.13333333333333333 saat


In [ ]:
"""v11 Colab execution cell. Completed arms are immutable and resumable."""


def arm_record_path(inst, arm):
    return OUTPUT_ROOT / "instances" / inst["name"] / arm / "result.json"


def validate_saved(inst, arm, record):
    assert record["complete"] and record["arm"] == arm and record["instance"] == inst["name"]
    assert record["xy_sha256"] == inst["xy_sha256"] and record["seeds"] == SEEDS
    assert record["budget_s"] == inst["budget_s"] and record["build"] == BUILD_INFO
    assert record["code_sha256"] == CODE_SHA256
    assert record["fraction"] == SPARSE_FRACTION and record["graph_k"] == GRAPH_K
    assert len(record["seed_costs"]) == len(SEEDS)
    assert min(record["seed_costs"]) == record["best_cost"]
    if arm in ("native_pop6", "native_pop7"):
        assert set(record["final_tour_sources"]) == {str(seed) for seed in SEEDS}
    else:
        assert len(record["first_tour_hashes"]) == len(SEEDS)
        assert len(record["live_candidate_hashes"]) == len(SEEDS)
        assert len(record["native_candidate_hashes"]) == len(SEEDS)
        audit = record["policy_audit"]
        assert len(audit["gate_sha256_per_seed"]) == len(SEEDS)
        assert audit["seed_ids"] == SEEDS
        assert audit["policy_version"] == POLICY_VERSION
        assert audit["decision_basis"] in ("fresh_inference", "cached_inference")
        if arm == "sham":
            assert audit["selected_actions_per_seed"] == [0] * len(SEEDS)
        else:
            assert audit["selected_actions_per_seed"] == audit["guided_actions_per_seed"]
        assert audit["selected_unique_undirected_edges_per_seed"] == audit["selected_actions_per_seed"]
    return record


def save_progress(state="running", last=None):
    completed = sum(arm_record_path(inst, arm).exists() for inst in INSTANCES for arm in ARMS)
    write_json_atomic(OUTPUT_ROOT / "progress.json",
                      {"state": state, "completed_arms": completed,
                       "total_arms": len(INSTANCES) * len(ARMS), "last": last,
                       "updated_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())})


new_arms = 0
save_progress()
for inst in INSTANCES:
    instance_dir = OUTPUT_ROOT / "instances" / inst["name"]
    instance_dir.mkdir(parents=True, exist_ok=True)
    coord_path = instance_dir / "xy.npz"
    if not coord_path.exists():
        temp = instance_dir / "xy.tmp.npz"
        np.savez_compressed(temp, xy=inst["xy"])
        os.replace(temp, coord_path)
    with np.load(coord_path, allow_pickle=False) as saved_xy:
        assert hashlib.sha256(np.ascontiguousarray(saved_xy["xy"]).tobytes()).hexdigest() == inst["xy_sha256"]
    local_problem = WORK_ROOT / "problems" / (inst["name"] + ".tsp")
    local_problem.parent.mkdir(exist_ok=True)
    T.write_tsp(local_problem, inst["xy"])
    for arm in arm_order(inst["name"]):
        record_path = arm_record_path(inst, arm)
        if record_path.exists():
            validate_saved(inst, arm, json.loads(record_path.read_text()))
            print("ATLA:", inst["name"], arm, flush=True)
            continue
        if MAX_NEW_ARMS is not None and new_arms >= MAX_NEW_ARMS:
            break
        attempt_id = uuid.uuid4().hex
        local_run = WORK_ROOT / "runs" / inst["name"] / arm / attempt_id
        local_run.parent.mkdir(parents=True, exist_ok=True)
        try:
            print("BAŞLA:", inst["name"], arm, "bütçe", inst["budget_s"], "sn", flush=True)
            if arm.startswith("native_"):
                raw = run_native_arm(arm, NATIVE_LKH, local_problem, inst["xy"],
                                     local_run, SEEDS, inst["budget_s"], T)
            else:
                # Arms whose model input is bitwise identical decide from the
                # first stored model output; every arm still runs inference.
                raw = run_guided_arm(arm, PATCHED_LKH, local_problem, inst["xy"],
                                     local_run, SEEDS, inst["budget_s"],
                                     MODEL, DEVICE, T, M,
                                     fraction=SPARSE_FRACTION, graph_k=GRAPH_K,
                                     decision_cache_dir=instance_dir / "decision_cache",
                                     model_identity=CHECKPOINT_SHA256)
            audit_rel = f"instances/{inst['name']}/{arm}/{attempt_id}"
            audit_dest = OUTPUT_ROOT / audit_rel
            shutil.copytree(local_run, audit_dest)
            record = {"schema": 1, "complete": True, "instance": inst["name"],
                      "group": inst["group"], "n": inst["n"],
                      "xy_sha256": inst["xy_sha256"], "arm": arm,
                      "seeds": SEEDS, "budget_s": inst["budget_s"],
                      "seed_costs": raw["seed_costs"], "best_cost": raw["best_cost"],
                      "wall_elapsed_s": raw["wall_elapsed_s"],
                      "start_epoch_s": raw["start_epoch_s"],
                      "deadline_epoch_s": raw["deadline_epoch_s"] if "deadline_epoch_s" in raw else raw["start_epoch_s"] + inst["budget_s"],
                      "first_costs": raw.get("first_costs"),
                      "first_tour_hashes": raw.get("first_tour_hashes"),
                      "live_candidate_hashes": raw.get("live_candidate_hashes"),
                      "native_candidate_hashes": raw.get("native_candidate_hashes"),
                      "policy_audit": raw.get("policy_audit"),
                      "guided_improvement_events": raw.get("guided_improvement_events"),
                      "native_improvement_events": raw.get("native_improvement_events"),
                      "final_tour_sources": raw.get("final_tour_sources"),
                      "event_flags": raw.get("event_flags"),
                      "decision_cache_files_written": raw.get("decision_cache_files_written"),
                      "fraction": SPARSE_FRACTION, "graph_k": GRAPH_K,
                      "build": BUILD_INFO, "code_sha256": CODE_SHA256,
                      "hardware": HARDWARE, "audit_relpath": audit_rel,
                      "attempt_id": attempt_id,
                      "completed_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
            validate_saved(inst, arm, record)
            write_json_atomic(record_path, record)
            new_arms += 1
            save_progress(last=inst["name"] + "/" + arm)
            print("BİTTİ:", inst["name"], arm, "en iyi", raw["best_cost"], flush=True)
        except Exception as exc:
            write_json_atomic(OUTPUT_ROOT / "errors" /
                              (inst["name"] + "_" + arm + "_" + attempt_id + ".json"),
                              {"instance": inst["name"], "arm": arm,
                               "error": repr(exc),
                               "utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())})
            save_progress(state="error", last=inst["name"] + "/" + arm)
            raise
    if MAX_NEW_ARMS is not None and new_arms >= MAX_NEW_ARMS:
        break

completed_count = sum(arm_record_path(inst, arm).exists() for inst in INSTANCES for arm in ARMS)
save_progress(state="complete" if completed_count == len(INSTANCES) * len(ARMS) else "partial")
print(f"Tamamlanan kol: {completed_count}/{len(INSTANCES) * len(ARMS)}. "
      f"Bu oturumdaki yeni kol: {new_arms}")


BAŞLA: augment_clu10k_000 random_add bütçe 120.0 sn
BİTTİ: augment_clu10k_000 random_add en iyi 33761035
BAŞLA: augment_clu10k_000 native_pop6 bütçe 120.0 sn
BİTTİ: augment_clu10k_000 native_pop6 en iyi 33760199
BAŞLA: augment_clu10k_000 frequency_add bütçe 120.0 sn
BİTTİ: augment_clu10k_000 frequency_add en iyi 33760675
BAŞLA: augment_clu10k_000 sham bütçe 120.0 sn
BİTTİ: augment_clu10k_000 sham en iyi 33760199
BAŞLA: augment_clu10k_000 native_pop7 bütçe 120.0 sn
BİTTİ: augment_clu10k_000 native_pop7 en iyi 33761415
BAŞLA: augment_clu10k_000 beta_add bütçe 120.0 sn
BİTTİ: augment_clu10k_000 beta_add en iyi 33760409
BAŞLA: augment_clu10k_001 native_pop6 bütçe 120.0 sn
BİTTİ: augment_clu10k_001 native_pop6 en iyi 16201186
BAŞLA: augment_clu10k_001 random_add bütçe 120.0 sn
BİTTİ: augment_clu10k_001 random_add en iyi 16203156
BAŞLA: augment_clu10k_001 native_pop7 bütçe 120.0 sn
BİTTİ: augment_clu10k_001 native_pop7 en iyi 16201768
BAŞLA: augment_clu10k_001 beta_add bütçe 120.0 sn
BİTTİ: 

In [ ]:
"""v11r2 Colab summary cell: mechanism measurements and guarded comparisons.

Primary endpoint (as in v9): the shortest valid tour among six seeds at the
deadline, compared by instance. Exact two-sided sign tests are descriptive
because the four instances were examined in earlier runs. Mean-of-six and
seed-level summaries, per-action mechanism rates and the trial-count ratio
against sham were added after the 2-vCPU v11 run and are descriptive.
"""

from math import comb


def read_key_value_metrics(path):
    values = {}
    for line in Path(path).read_text().splitlines():
        words = line.split()
        if len(words) != 2 or words[0] in values:
            raise ValueError(f"Bozuk telemetri satırı: {path}: {line!r}")
        values[words[0]] = float(words[1]) if "." in words[1] else int(words[1])
    return values


def exact_sign_p(wins, losses):
    n, k = wins + losses, min(wins, losses)
    return 1.0 if n == 0 else min(1.0, 2.0 * sum(comb(n, j) for j in range(k + 1)) / 2 ** n)


def comparison_summary(frame, treatment, control, basis, column="{}"):
    if frame.empty:
        return None
    t = frame[column.format(treatment)].to_numpy(dtype=np.float64)
    c = frame[column.format(control)].to_numpy(dtype=np.float64)
    rel = 100.0 * (t - c) / c
    wins, losses = int((rel < 0).sum()), int((rel > 0).sum())
    return {"treatment": treatment, "control": control, "basis": basis,
            "n_instances": len(rel), "wins": wins, "losses": losses,
            "ties": int((rel == 0).sum()),
            "mean_relative_pct": float(rel.mean()),
            "median_relative_pct": float(np.median(rel)),
            "exact_sign_p_descriptive": exact_sign_p(wins, losses)}


patched_arms = ["sham", "random_add", "frequency_add", "beta_add"]
guided_arms = ["random_add", "frequency_add", "beta_add"]
rows, mechanism_rows, target_rows, frequency_rows, event_rows = [], [], [], [], []
seed_rows = []
summary_dir = OUTPUT_ROOT / "summary"
summary_dir.mkdir(exist_ok=True)

for inst in INSTANCES:
    records = {}
    for arm in ARMS:
        path = arm_record_path(inst, arm)
        if path.exists():
            records[arm] = validate_saved(inst, arm, json.loads(path.read_text()))
    if len(records) != len(ARMS):
        continue

    row = {"instance": inst["name"], "group": inst["group"],
           "n": inst["n"], "budget_s": inst["budget_s"],
           "mixed_hardware": len({json.dumps(records[arm]["hardware"], sort_keys=True)
                                   for arm in ARMS}) > 1}
    for arm in ARMS:
        row[arm] = records[arm]["best_cost"]
        row["mean6_" + arm] = float(np.mean(records[arm]["seed_costs"]))
    audits = {arm: records[arm]["policy_audit"] for arm in patched_arms}
    row["first_tours_match"] = all(
        len({records[arm]["first_tour_hashes"][i] for arm in patched_arms}) == 1
        for i in range(len(SEEDS)))
    row["raw_first_tours_match"] = all(
        len({audits[arm]["first_tour_raw_sha256"][i] for arm in patched_arms}) == 1
        for i in range(len(SEEDS)))
    row["native_candidates_match"] = all(
        len({records[arm]["native_candidate_hashes"][i] for arm in patched_arms}) == 1
        for i in range(len(SEEDS)))
    row["live_candidates_match"] = all(
        len({records[arm]["live_candidate_hashes"][i] for arm in patched_arms}) == 1
        for i in range(len(SEEDS)))
    row["gates_match"] = all(
        len({audits[arm]["gate_sha256_per_seed"][i] for arm in patched_arms}) == 1
        for i in range(len(SEEDS)))
    row["model_predictions_match"] = len({
        audits[arm]["model_probability_sha256"] for arm in patched_arms}) == 1
    row["fresh_predictions_match"] = len({
        audits[arm]["fresh_model_probability_sha256"] for arm in patched_arms}) == 1
    row["fresh_inference_arms"] = sum(audits[arm]["decision_basis"] == "fresh_inference"
                                      for arm in patched_arms)
    differences = [audits[arm]["fresh_vs_decision_max_abs_probability_difference"]
                   for arm in patched_arms]
    row["max_fresh_vs_decision_probability_difference"] = (
        None if any(value is None for value in differences) else max(differences))
    row["planned_actions_match"] = len({
        json.dumps(audits[arm]["all_guided_action_sha256_per_seed"], sort_keys=True)
        for arm in patched_arms}) == 1
    row["capacities_match"] = all(
        len({audits[arm]["selected_actions_per_seed"][i] for arm in guided_arms}) == 1
        for i in range(len(SEEDS)))
    row["undirected_capacities_match"] = all(
        len({audits[arm]["selected_unique_undirected_edges_per_seed"][i]
             for arm in guided_arms}) == 1 for i in range(len(SEEDS)))
    row["matched_policy_state"] = all(row[key] for key in
                                      ("first_tours_match", "raw_first_tours_match",
                                       "native_candidates_match", "live_candidates_match",
                                       "model_predictions_match", "planned_actions_match",
                                       "gates_match", "capacities_match",
                                       "undirected_capacities_match"))
    rows.append(row)

    for index, seed in enumerate(SEEDS):
        seed_rows.append({"instance": inst["name"], "seed": seed,
                          **{arm: records[arm]["seed_costs"][index] for arm in ARMS}})

    for arm in patched_arms:
        record = records[arm]
        for index, seed in enumerate(SEEDS):
            base = OUTPUT_ROOT / record["audit_relpath"] / f"seed_{seed}"
            metrics = read_key_value_metrics(base / "telemetry.txt")
            applied = read_key_value_metrics(base / "applied.stats")
            selected = int(record["policy_audit"]["selected_actions_per_seed"][index])
            if int(metrics["action_count"]) != selected:
                raise AssertionError(f"Seçilen/uygulanan kenar sayısı farklı: {inst['name']}/{arm}/{seed}")
            events = record["guided_improvement_events"][str(seed)]
            mechanism_rows.append({"instance": inst["name"], "arm": arm,
                                   "seed": seed, "selected_actions": selected,
                                   "first_trial_end_s": events[0]["elapsed_s"],
                                   "barrier_wait_s": applied["wait_wall_s"],
                                   "telemetry_age_at_end_s": max(0.0, min(
                                       float(record["deadline_epoch_s"]),
                                       float(record["start_epoch_s"]) + float(record["wall_elapsed_s"]))
                                       - float(metrics["telemetry_snapshot_epoch_s"])),
                                   **metrics})

    for arm in ARMS:
        record = records[arm]
        event_map = (record["native_improvement_events"] if arm.startswith("native_")
                     else record["guided_improvement_events"])
        flags = record.get("event_flags") or {}
        for seed in SEEDS:
            trace = event_map[str(seed)]
            event_rows.append({"instance": inst["name"], "arm": arm, "seed": seed,
                               "events": len(trace),
                               "first_event_s": trace[0]["elapsed_s"],
                               "last_event_s": trace[-1]["elapsed_s"],
                               "events_after_deadline": sum(event["elapsed_s"] > inst["budget_s"]
                                                            for event in trace),
                               "final_tour_source": (record["final_tour_sources"][str(seed)]
                                                     if arm.startswith("native_") else "atomic_hook_best_tour"),
                               "flags": ";".join(flags.get(str(seed), []))})

    frequency_audit = records["beta_add"]["policy_audit"]
    for arm in guided_arms:
        for index, seed in enumerate(SEEDS):
            values = frequency_audit["all_guided_selected_ensemble_frequency_per_seed"][arm][index]
            frequency_rows.append({"instance": inst["name"], "arm": arm,
                                   "seed": seed, "selected_actions": len(values),
                                   "zero_frequency_actions": sum(value == 0 for value in values),
                                   "mean_ensemble_frequency": float(np.mean(values)) if values else np.nan})

    target = TARGET_COSTS[inst["name"]]
    for arm in ARMS:
        record = records[arm]
        event_map = (record["native_improvement_events"] if arm.startswith("native_")
                     else record["guided_improvement_events"])
        observed = [float(event["elapsed_s"])
                    for seed in SEEDS for event in event_map[str(seed)]
                    if int(event["cost"]) <= target and float(event["elapsed_s"]) <= inst["budget_s"]]
        target_rows.append({"instance": inst["name"], "arm": arm,
                            "target_cost_v10_native_pop6_best_of_six": target,
                            "reached": bool(observed),
                            "earliest_elapsed_s": min(observed) if observed else np.nan,
                            "trace_kind": "approximate_native_file_poll" if arm.startswith("native_")
                            else "exact_patched_best_event"})

table = pd.DataFrame(rows)
seed_table = pd.DataFrame(seed_rows)
mechanism_table = pd.DataFrame(mechanism_rows)
target_table = pd.DataFrame(target_rows)
frequency_table = pd.DataFrame(frequency_rows)
event_table = pd.DataFrame(event_rows)
table.to_csv(summary_dir / "complete_instances.csv", index=False)
seed_table.to_csv(summary_dir / "seed_costs.csv", index=False)
mechanism_table.to_csv(summary_dir / "candidate_mechanism.csv", index=False)
target_table.to_csv(summary_dir / "time_to_target_exploratory.csv", index=False)
frequency_table.to_csv(summary_dir / "selected_edge_frequencies.csv", index=False)
event_table.to_csv(summary_dir / "improvement_trace_quality.csv", index=False)

same_hardware = table.loc[~table["mixed_hardware"]].copy() if not table.empty else table
matched = same_hardware.loc[same_hardware["matched_policy_state"]].copy() if not same_hardware.empty else same_hardware
comparison_columns = ["treatment", "control", "basis", "n_instances", "wins",
                      "losses", "ties", "mean_relative_pct", "median_relative_pct",
                      "exact_sign_p_descriptive"]
endpoint_rows, matched_rows, mean_rows = [], [], []
for control in ("native_pop6", "native_pop7", "sham", "random_add", "frequency_add"):
    result = comparison_summary(same_hardware, "beta_add", control,
                                "primary endpoint: best of six seeds")
    if result is not None:
        endpoint_rows.append(result)
    result = comparison_summary(same_hardware, "beta_add", control,
                                "descriptive: mean of six seed costs", "mean6_{}")
    if result is not None:
        mean_rows.append(result)
for treatment, control in (("sham", "native_pop6"), ("native_pop7", "native_pop6"),
                           ("frequency_add", "sham"), ("random_add", "sham"),
                           ("frequency_add", "random_add")):
    result = comparison_summary(same_hardware, treatment, control,
                                "context: best of six seeds")
    if result is not None:
        endpoint_rows.append(result)
for control in ("sham", "random_add", "frequency_add"):
    result = comparison_summary(matched, "beta_add", control,
                                "all six raw first tours/candidate states/decision inputs/actions/gates/capacities match")
    if result is not None:
        matched_rows.append(result)
endpoint_table = pd.DataFrame(endpoint_rows + mean_rows, columns=comparison_columns)
matched_table = pd.DataFrame(matched_rows, columns=comparison_columns)
endpoint_table.to_csv(summary_dir / "endpoint_comparisons.csv", index=False)
matched_table.to_csv(summary_dir / "matched_policy_comparisons.csv", index=False)

seed_level_rows = []
if not seed_table.empty:
    for treatment, control in (("beta_add", "sham"), ("beta_add", "random_add"),
                               ("beta_add", "frequency_add"), ("beta_add", "native_pop6")):
        rel = 100.0 * (seed_table[treatment] - seed_table[control]) / seed_table[control]
        seed_level_rows.append({"treatment": treatment, "control": control,
                                "seed_pairs": len(rel), "wins": int((rel < 0).sum()),
                                "losses": int((rel > 0).sum()), "ties": int((rel == 0).sum()),
                                "mean_relative_pct": float(rel.mean()),
                                "note": "descriptive; seeds are not independent units"})
seed_level_table = pd.DataFrame(seed_level_rows)
seed_level_table.to_csv(summary_dir / "seed_level_comparisons.csv", index=False)

mechanism_columns = ["selected_actions", "action_candidate_enumerations",
                     "distinct_actions_enumerated", "action_accepted_edge_entries",
                     "distinct_actions_entering_accepted_tours",
                     "top_level_action_enumerations",
                     "top_level_action_first_gate_passes", "top_level_action_positive_g1",
                     "action_incumbent_entries", "distinct_actions_ever_in_better_tour",
                     "current_better_tour_actions", "better_tour_events_after_action",
                     "trials_completed_after_action", "total_candidate_enumerations",
                     "slot_checks", "slot_restorations", "slot_restoration_displacement",
                     "slot_found_ahead", "slot_missing"]
if mechanism_table.empty:
    mechanism_aggregate = pd.DataFrame()
    rate_table = pd.DataFrame()
else:
    mechanism_aggregate = mechanism_table.groupby("arm", as_index=False)[mechanism_columns].sum()
    guided_totals = mechanism_aggregate.set_index("arm").loc[guided_arms]
    rate_table = pd.DataFrame({
        "enumerations_per_action": guided_totals.action_candidate_enumerations / guided_totals.selected_actions,
        "first_gate_passes_per_action": guided_totals.top_level_action_first_gate_passes / guided_totals.selected_actions,
        "accepted_entries_per_action": guided_totals.action_accepted_edge_entries / guided_totals.selected_actions,
        "pct_actions_ever_accepted": 100 * guided_totals.distinct_actions_entering_accepted_tours / guided_totals.selected_actions,
        "incumbent_entries_per_1000_actions": 1000 * guided_totals.action_incumbent_entries / guided_totals.selected_actions,
        "pct_actions_in_final_incumbent": 100 * guided_totals.current_better_tour_actions / guided_totals.selected_actions,
        "slot_restorations_per_action": guided_totals.slot_restorations / guided_totals.selected_actions,
    })
    sham_trials = mechanism_table[mechanism_table.arm == "sham"].set_index(["instance", "seed"])
    for arm in guided_arms:
        arm_trials = mechanism_table[mechanism_table.arm == arm].set_index(["instance", "seed"])
        ratio = (arm_trials.trials_completed_after_action /
                 sham_trials.trials_completed_after_action.replace(0, np.nan))
        rate_table.loc[arm, "median_trials_ratio_vs_sham"] = float(ratio.median())
    rate_table = rate_table.reset_index().rename(columns={"index": "arm"})
mechanism_aggregate.to_csv(summary_dir / "candidate_mechanism_totals.csv", index=False)
rate_table.to_csv(summary_dir / "candidate_mechanism_rates.csv", index=False)

write_json_atomic(summary_dir / "pilot_note.json", {
    "analysis_revision": "v11r2-state-guarded-mechanism-2",
    "completed_instances": len(table), "planned_instances": len(INSTANCES),
    "same_hardware_instances": len(same_hardware),
    "matched_policy_instances": len(matched),
    "hardware": HARDWARE,
    "interpretation": ("Reused-case mechanism pilot. Primary endpoint: best of six seeds by instance; "
                       "sign-test p-values are descriptive. Mean-of-six, seed-level and per-action "
                       "rates are descriptive. Patched policy comparisons require exact six-seed state, "
                       "decision input, gate and capacity matching. Native traces are approximate file "
                       "polls; patched best.events are exact. Incumbent entries are observational, not "
                       "causal. No manuscript superiority claim.")
})

print("Tamamlanan örnek:", len(table), "/", len(INSTANCES),
      "; tam eşleşmiş politika örneği:", len(matched))
display(table)
display(endpoint_table)
display(matched_table)
display(seed_level_table)
display(mechanism_aggregate)
display(rate_table)
display(frequency_table.groupby("arm")[["selected_actions", "zero_frequency_actions"]].sum()
        if not frequency_table.empty else frequency_table)
display(event_table.groupby("arm")[["events", "events_after_deadline"]].sum()
        if not event_table.empty else event_table)
display(target_table)


Tamamlanan örnek: 4 / 4 ; tam eşleşmiş politika örneği: 4


,instance,group,n,budget_s,mixed_hardware,native_pop6,mean6_native_pop6,native_pop7,mean6_native_pop7,sham,...,live_candidates_match,gates_match,model_predictions_match,fresh_predictions_match,fresh_inference_arms,max_fresh_vs_decision_probability_difference,planned_actions_match,capacities_match,undirected_capacities_match,matched_policy_state
0,augment_clu10k_000,clu10k,10000,120.0,False,33760199,3.376252e+07,33761415,3.376198e+07,33760199,...,True,True,True,True,1,0.0,True,True,True,True
1,augment_clu10k_001,clu10k,10000,120.0,False,16201186,1.620333e+07,16201768,1.620405e+07,16201411,...,True,True,True,True,1,0.0,True,True,True,True
2,augment_uni10k_000,uni10k,10000,120.0,False,72087781,7.209803e+07,72092113,7.209732e+07,72088540,...,True,True,True,True,1,0.0,True,True,True,True
3,augment_uni10k_001,uni10k,10000,120.0,False,71648570,7.165705e+07,71649539,7.166048e+07,71653253,...,True,True,True,True,1,0.0,True,True,True,True


,treatment,control,basis,n_instances,wins,losses,ties,mean_relative_pct,median_relative_pct,exact_sign_p_descriptive
0,beta_add,native_pop6,primary endpoint: best of six seeds,4,1,3,0,0.000700,0.000776,0.625
1,beta_add,native_pop7,primary endpoint: best of six seeds,4,3,1,0,-0.002939,-0.001701,0.625
2,beta_add,sham,primary endpoint: best of six seeds,4,2,2,0,-0.001545,-0.001953,1.000
3,beta_add,random_add,primary endpoint: best of six seeds,4,4,0,0,-0.004903,-0.005161,0.125
4,beta_add,frequency_add,primary endpoint: best of six seeds,4,3,1,0,-0.004817,-0.006044,0.625
5,sham,native_pop6,context: best of six seeds,4,0,3,1,0.002244,0.001221,0.250
6,native_pop7,native_pop6,context: best of six seeds,4,0,4,0,0.003639,0.003597,0.125
7,frequency_add,sham,context: best of six seeds,4,1,3,0,0.003273,0.003553,0.625
8,random_add,sham,context: best of six seeds,4,1,3,0,0.003359,0.001838,0.625
9,frequency_add,random_add,context: best of six seeds,4,2,2,0,-0.000085,0.001714,1.000


,treatment,control,basis,n_instances,wins,losses,ties,mean_relative_pct,median_relative_pct,exact_sign_p_descriptive
0,beta_add,sham,all six raw first tours/candidate states/decis...,4,2,2,0,-0.001545,-0.001953,1.000
1,beta_add,random_add,all six raw first tours/candidate states/decis...,4,4,0,0,-0.004903,-0.005161,0.125
2,beta_add,frequency_add,all six raw first tours/candidate states/decis...,4,3,1,0,-0.004817,-0.006044,0.625


,treatment,control,seed_pairs,wins,losses,ties,mean_relative_pct,note
0,beta_add,sham,24,12,12,0,-0.000640,descriptive; seeds are not independent units
1,beta_add,random_add,24,18,6,0,-0.008174,descriptive; seeds are not independent units
2,beta_add,frequency_add,24,16,8,0,-0.004963,descriptive; seeds are not independent units
3,beta_add,native_pop6,24,11,13,0,0.001640,descriptive; seeds are not independent units


,arm,selected_actions,action_candidate_enumerations,distinct_actions_enumerated,action_accepted_edge_entries,distinct_actions_entering_accepted_tours,top_level_action_enumerations,top_level_action_first_gate_passes,top_level_action_positive_g1,action_incumbent_entries,distinct_actions_ever_in_better_tour,current_better_tour_actions,better_tour_events_after_action,trials_completed_after_action,total_candidate_enumerations,slot_checks,slot_restorations,slot_restoration_displacement,slot_found_ahead,slot_missing
0,beta_add,60000,2239682323,60000,1140309,59480,12787136,6933761,6933761,671,566,251,931,3444,54031693601,3444,54100,56117,35212,0
1,frequency_add,60000,2273987471,60000,964797,59249,12847959,9023316,9023316,483,415,224,881,3216,54396812361,3216,54321,56215,28754,0
2,random_add,60000,2206476903,60000,644651,58144,12479171,5427400,5427400,168,131,56,884,3607,53623635541,3607,53537,55541,8486,0
3,sham,0,0,0,0,0,0,0,0,0,0,0,993,4100,53956597353,4100,0,0,0,0


,arm,enumerations_per_action,first_gate_passes_per_action,accepted_entries_per_action,pct_actions_ever_accepted,incumbent_entries_per_1000_actions,pct_actions_in_final_incumbent,slot_restorations_per_action,median_trials_ratio_vs_sham
0,random_add,36774.615050,90.456667,10.744183,96.906667,2.800000,0.093333,0.892283,0.890397
1,frequency_add,37899.791183,150.388600,16.079950,98.748333,8.050000,0.373333,0.905350,0.799610
2,beta_add,37328.038717,115.562683,19.005150,99.133333,11.183333,0.418333,0.901667,0.819331


,selected_actions,zero_frequency_actions
arm,,
beta_add,60000,59105
frequency_add,60000,59105
random_add,60000,59819


,events,events_after_deadline
arm,,
beta_add,955,0
frequency_add,905,0
native_pop6,1042,0
native_pop7,881,0
random_add,908,0
sham,1017,0


,instance,arm,target_cost_v10_native_pop6_best_of_six,reached,earliest_elapsed_s,trace_kind
0,augment_clu10k_000,native_pop6,33760199,True,106.557510,approximate_native_file_poll
1,augment_clu10k_000,native_pop7,33760199,False,NaN,approximate_native_file_poll
2,augment_clu10k_000,sham,33760199,True,115.237334,exact_patched_best_event
3,augment_clu10k_000,random_add,33760199,False,NaN,exact_patched_best_event
4,augment_clu10k_000,frequency_add,33760199,False,NaN,exact_patched_best_event
5,augment_clu10k_000,beta_add,33760199,False,NaN,exact_patched_best_event
6,augment_clu10k_001,native_pop6,16201186,True,105.913180,approximate_native_file_poll
7,augment_clu10k_001,native_pop7,16201186,False,NaN,approximate_native_file_poll
8,augment_clu10k_001,sham,16201186,False,NaN,exact_patched_best_event
9,augment_clu10k_001,random_add,16201186,False,NaN,exact_patched_best_event


In [ ]:
def checked_zip(name, include_audit):
    local = WORK_ROOT / name
    temp = WORK_ROOT / (name + ".tmp")
    files = []
    with zipfile.ZipFile(temp, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6, allowZip64=True) as archive:
        for path in sorted(OUTPUT_ROOT.rglob("*")):
            if not path.is_file() or path.suffix == ".zip" or path.name.endswith(".tmp") or path.name.endswith(".tmp.npz"):
                continue
            rel = path.relative_to(OUTPUT_ROOT)
            if not include_audit and ("policy_audit" in rel.parts or "decision_cache" in rel.parts or path.name in
                                      {"xy.npz", "native.candidates", "live.candidates", "post.candidates", "first.tour", "best.tour", "action.txt", "stdout.txt", "stderr.txt", "policy_arrays.npz"}):
                continue
            archive.write(path, str(rel), compress_type=zipfile.ZIP_STORED if path.suffix == ".npz" else zipfile.ZIP_DEFLATED)
            files.append(str(rel))
    with zipfile.ZipFile(temp) as check:
        assert check.testzip() is None and len(check.namelist()) == len(files)
    os.replace(temp, local)
    target = OUTPUT_ROOT / "archives" / name
    tmp_target = target.with_name(target.name + ".tmp")
    shutil.copy2(local, tmp_target)
    os.replace(tmp_target, target)
    assert sha256_file(local) == sha256_file(target)
    return {"local": str(local), "drive": str(target), "sha256": sha256_file(local),
            "bytes": local.stat().st_size, "files": len(files)}

complete = len(rows) == len(INSTANCES)
audit_zip = checked_zip(EXPERIMENT_NAME + "_audit.zip", True) if complete else None
results_zip = checked_zip(EXPERIMENT_NAME + "_results.zip", False)
write_json_atomic(OUTPUT_ROOT / "archives" / "archive_info.json",
                  {"complete": complete, "results": results_zip, "audit": audit_zip})
print("Sonuç ZIP'i:", results_zip)
if complete:
    print("Tam denetim ZIP'i Drive'da:", audit_zip)
    from google.colab import files
    files.download(results_zip["local"])
else:
    print(f"Ara sonuçlar Drive'da; tamamlanan örnek {len(rows)}/{len(INSTANCES)}. Yeniden çalıştırınca devam eder.")


Sonuç ZIP'i: {'local': '/content/betatour_v11r2_candidate_work/v11r2_pop6_candidate_mechanism_results.zip', 'drive': '/content/drive/MyDrive/BetaTour/v11r2_pop6_candidate_mechanism/archives/v11r2_pop6_candidate_mechanism_results.zip', 'sha256': 'c4e7f22afe8ee4aeea7e4634be66ab1d469762ad1b91fce0d28161197907c843', 'bytes': 18309325, 'files': 632}
Tam denetim ZIP'i Drive'da: {'local': '/content/betatour_v11r2_candidate_work/v11r2_pop6_candidate_mechanism_audit.zip', 'drive': '/content/drive/MyDrive/BetaTour/v11r2_pop6_candidate_mechanism/archives/v11r2_pop6_candidate_mechanism_audit.zip', 'sha256': '2067dae2312b30901221227eac50b2454476ce3626e02c8128ce1e13879e0767', 'bytes': 116199933, 'files': 1552}


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>